# 도쿄일렉트론코리아 RAG 교육 - 기초 과정

## 2일차 목차

| 번호 | 주제 | 핵심 내용 |
|---|---|---|
| 1 | 빠른 복습: 기본 RAG 흐름 | 기본 RAG 흐름과 개선 지점 정리 |
| 2 | Advanced RAG 기법 | Multi-Query, HyDE, Semantic Routing(RAG 라우팅), Hybrid Retrieval, Reranking, Compression, Multi-Representation |
| 3 | 참고자료 | 주요 기법의 참고 논문 |


## 1. 빠른 복습: 기본 RAG 흐름

<img src="image/RAG.png" width="640">

2일차의 Advanced RAG는 이 기본 흐름을 버리는 것이 아니라,  
각 단계(Query, Retrieval, Post-Retrieval, Representation)를 더 정교하게 다듬는 과정입니다.

## 2. Advanced RAG 기법

1일차에서 본 기본 RAG는 `검색 -> 답변 생성` 흐름을 이해하는 출발점이었습니다.  
2일차에서는 그 흐름을 버리는 것이 아니라, 검색 품질을 높이기 위해 각 단계를 더 정교하게 다듬는 방법을 봅니다.

오늘 볼 관점은 아래 네 가지입니다.
- **Pre-Retrieval**: 검색 전에 질문이나 입력을 더 잘 찾히는 형태로 바꾸는 단계
- **Retrieval**: 정답 가능성이 있는 문서를 최대한 빠뜨리지 않고 모으는 단계
- **Post-Retrieval**: 모아온 후보를 다시 읽고, 중복·잡음·불필요한 길이를 줄이는 단계
- **Representation**: 같은 문서를 어떤 표현으로 저장하고 검색할지 정하는 단계

정리하면,
- `Pre-Retrieval`은 **찾기 전에 질문을 다듬는 단계**이고
- `Post-Retrieval`은 **찾은 뒤 문서를 다듬는 단계**입니다.
- `Representation`은 **검색이 잘 되도록 문서를 다시 표현하는 단계**입니다.

핵심은 **어떤 기법이 어느 단계의 검색 문제를 줄이는지** 흐름으로 보는 것입니다.

<img src="image/paradigms_of_RAG.png" width="600">

### 2.1 Query Transformation 확장: Multi-Query Retrieval

이 구간은 **Pre-Retrieval**에 해당합니다.  
즉, 검색기를 바꾸기 전에 **질문 자체를 더 잘 찾히는 형태로 바꾸는 단계**입니다.

1일차에서는 하나의 질문을 더 또렷하게 바꾸는 **Query Rewriting**을 살펴봤습니다.  
여기서는 그 다음 단계로, 하나의 질문을 **여러 검색 질문으로 확장**하는 방법을 봅니다.

**Multi-Query Retrieval**은 하나의 질문에서 여러 검색 쿼리를 만든 뒤,  
각 쿼리를 독립적으로 검색하고 결과를 합쳐 중복을 정리하는 방식입니다.

흐름은 단순합니다.

1. **LLM이 원 질문을 여러 검색 질문으로 생성**
2. **각 쿼리를 Retriever에 독립적으로 질의**
3. **결과를 합치고 같은 문서는 한 번만 남김**

이 방식은 결과가 항상 더 좋아진다기보다,  
**원 질문 하나로는 놓칠 수 있는 관련 후보를 더 넓게 모으는 데** 의미가 있습니다.


##### 기본 예시: 질문 생성과 검색 결과 비교

여기서는 **원 질문 상위 결과**와 **확장 질문을 합친 뒤의 상위 결과**를 비교합니다.  
핵심은 정답을 바로 맞히는지보다, **관련 후보를 더 넓게 모으는지** 보는 것입니다.


> 참고: `Multi-Query Retrieval`과 `질문 분해(Query Decomposition)`는 닮았지만 완전히 같지는 않습니다.
> `Multi-Query`는 **같은 질문을 여러 검색 표현으로 넓히는 것**에 가깝고, `질문 분해`는 **복합 질문을 하위 질문으로 나눠 각각 해결한 뒤 다시 합치는 것**에 가깝습니다.
> 이 과정에서는 둘을 완전히 분리해 길게 다루기보다, 먼저 `Multi-Query` 안에서 연결된 개념으로 이해합니다.


**모델 및 벡터DB 준비**

공통으로 사용할 PDF, 임베딩 모델, ChromaDB 저장소, 기본 retriever를 준비합니다.


In [ ]:
import os  # 파일 경로와 환경 변수를 다룰 때 쓰는 표준 라이브러리입니다.
import warnings  # 학습 흐름과 무관한 경고를 숨길 때 쓰는 표준 라이브러리입니다.

warnings.filterwarnings("ignore")  # 학습 흐름과 무관한 경고는 모두 숨깁니다.

from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션에 불러옵니다.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 함께 불러옵니다.
from langchain_chroma import Chroma  # ChromaDB를 LangChain 벡터 저장소로 사용합니다.
from langchain_community.document_loaders import PyPDFLoader  # PDF를 페이지 단위 문서로 읽어옵니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥을 최대한 살리며 긴 문서를 여러 청크로 나눕니다.
from pdf_utils import clean_pdf_documents  # PDF 정제 공통 유틸을 불러옵니다.

load_dotenv(".env", override=True)  # 예제 실행에 필요한 API 키를 현재 세션에 반영합니다.

PDF_PATH = "data/금융투자협회_투자길라잡이_2018.pdf"  # 2일차 예제의 공통 원본 문서입니다.
DB_PATH = "./chroma_index_kofia_guide_cleaned"  # 정제한 문서를 저장한 ChromaDB 경로입니다.
COLLECTION_NAME = "kofia_guide_cleaned"  # ChromaDB 컬렉션 이름입니다.

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 문서와 질문을 같은 벡터 공간으로 바꿉니다.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 같은 질문이면 최대한 같은 답을 내도록 온도를 0으로 둡니다.

loader = PyPDFLoader(PDF_PATH)  # Hybrid 예제가 같은 청킹 기준을 보도록 원문을 다시 읽습니다.
docs = clean_pdf_documents(loader.load())  # 머리말, 쪽번호, 깨진 글리프를 정리합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 청크 하나의 최대 글자 수 기준입니다.
    chunk_overlap=100,  # 앞뒤 청크가 이어지도록 겹쳐 넣는 글자 수입니다.
    separators=["\n\n", ".", " ", ""],  # 문단, 문장, 단어 순서로 나눌 위치를 찾습니다.
)
split_docs = text_splitter.split_documents(docs)  # 정제된 PDF를 검색용 청크 리스트로 나눕니다.
for idx, d in enumerate(split_docs):
    d.metadata["id"] = idx  # Dense/BM25 검색기가 같은 청크 ID를 공유하게 맞춥니다.

if os.path.exists(DB_PATH):
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,  # 디스크 안에서 사용할 ChromaDB 컬렉션 이름입니다.
        embedding_function=embeddings,  # 질문을 같은 임베딩 방식으로 바꿔 검색합니다.
        persist_directory=DB_PATH,  # 이미 만들어 둔 벡터 DB가 저장된 폴더입니다.
    )
else:
    vectorstore = Chroma.from_documents(
        documents=split_docs,  # 처음 실행할 때 저장할 검색용 청크들입니다.
        embedding=embeddings,  # 각 청크를 벡터로 바꿀 임베딩 모델입니다.
        persist_directory=DB_PATH,  # 새로 만든 벡터 DB를 저장할 폴더입니다.
        collection_name=COLLECTION_NAME,  # 위 폴더 안에서 사용할 컬렉션 이름입니다.
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 여러 검색 비교에서 쓰기 좋도록 상위 6개를 가져옵니다.
print("공통 예제 준비 완료")
print("정제된 청크 수:", len(split_docs))


**Multi-Query 질문 생성 준비**

원 질문을 여러 검색 질문으로 바꾸는 프롬프트를 만듭니다.  
여기서는 모델이 만든 질문을 그대로 확인하는 데 집중합니다.


In [ ]:
from typing import List  # 함수 입력과 출력 타입을 읽기 쉽게 적을 때 씁니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.
from langchain_core.documents import Document  # 텍스트와 메타데이터를 함께 담는 문서 객체입니다.

# 한 질문을 서로 다른 검색 표현 2개로 넓히는 프롬프트입니다.
multi_query_prompt = PromptTemplate.from_template("""
당신은 금융투자협회의 투자자 안내서에서 정보를 찾는 검색 봇입니다.
사용자의 질문을 바탕으로, 실제 문서 본문에 등장할 법한 **구체적인 검색 쿼리 2개**를 생성하세요.

[작성 가이드]
1. 질문에 답하지 말고 **검색창에 넣을 문장형 쿼리**만 출력할 것
2. 두 질문은 같은 주제를 유지하되, 표현이나 초점을 조금씩 다르게 할 것
3. 문서에 없는 다른 주제로 퍼지지 말 것
4. 짧은 구어체보다 문서 본문에 가까운 표현으로 바꿀 것
5. 결과는 불릿 2개로만 출력할 것

원본 질문: {question}

출력 예시:
- 고객 동의 없이 직원이 매매한 경우 어떤 분쟁으로 보는가?
- 위탁받지 않고 매매한 경우 임의매매로 판단하는 기준은 무엇인가?
""")

# 각 쿼리 결과를 순서대로 합치고, 같은 문서는 한 번만 남깁니다.
def merge_multi_query_results(result_lists: List[List[Document]]) -> List[Document]:  # 여러 질문 결과를 하나로 합칩니다.
    merged_docs = []  # 최종 비교에 보여 줄 문서를 모읍니다.
    seen_ids = set()  # 이미 본 문서는 중복 추가하지 않기 위해 기록합니다.

    for docs in result_lists:
        for doc in docs:
            doc_id = doc.metadata.get("id", doc.page_content)  # 문서마다 비교 기준이 되는 ID를 꺼냅니다.
            if doc_id in seen_ids:
                continue
            seen_ids.add(doc_id)
            merged_docs.append(doc)

    return merged_docs

# 원 질문은 일부러 구어체로 두고, 확장 질문이 어떻게 만들어지는지 봅니다.
question = "고객 동의 없이 직원이 알아서 매매하면 어떤 분쟁이 되는가?"
multi_query_chain = multi_query_prompt | llm | StrOutputParser()  # 프롬프트 결과를 문자열로 받습니다.
raw_queries = multi_query_chain.invoke({"question": question})  # 모델이 만든 검색 질문 문자열입니다.

print(raw_queries)


**원 질문과 확장 질문 결과 비교**

위 출력에서 확인한 검색 질문을 그대로 리스트에 적어 검색합니다.  
자동 파싱 코드는 넣지 않습니다. 여기서는 검색 질문이 달라졌을 때 검색 결과가 어떻게 달라지는지만 봅니다.


In [ ]:
# 검색 결과를 짧게 비교 출력합니다.
def preview_docs(label: str, docs: List[Document], max_items: int = 3):  # 검색 결과를 짧게 미리 봅니다.
    print(f"\n=== {label} ===")
    for i, doc in enumerate(docs[:max_items], 1):
        page = doc.metadata.get("page")
        page_text = f", page={page + 1}" if isinstance(page, int) else ""
        print(f"[{i}] id={doc.metadata.get('id')}{page_text}")
        print(doc.page_content[:180])  # 본문 앞부분만 잘라서 비교합니다.
        print("---")

# 원 질문 검색과 확장 질문 검색 결과를 나란히 비교합니다.
expanded_queries = [
    "고객의 동의 없이 직원이 매매를 수행한 경우 어떤 분쟁으로 분류되는가?",
    "고객의 명시적 동의 없이 직원이 매매한 행위를 임의매매로 판단하는 구체적 기준은 무엇인가?",
]

original_docs = retriever.invoke(question)  # 원 질문만 넣었을 때의 결과입니다.
expanded_result_lists = [retriever.invoke(query)[:2] for query in expanded_queries]  # 확장 질문마다 상위 2개씩 가져옵니다.
merged_docs = merge_multi_query_results(expanded_result_lists)  # 여러 질문 결과를 중복 없이 합칩니다.

preview_docs("원 질문 검색 결과", original_docs, max_items=2)
preview_docs("확장 질문 병합 결과", merged_docs, max_items=3)

##### 실습 문제 1: 관점별 검색 질문 프롬프트 만들기

고객 문의 RAG에서는 하나의 질문을 한 표현으로만 검색하면 필요한 근거를 놓칠 수 있습니다.  
이 문제에서는 LLM이 같은 질문을 **서로 다른 검색 관점의 질문**으로 바꾸도록 프롬프트를 설계합니다.

여기서 중요한 것은 검색 질문을 사람이 직접 잘 쓰는 것이 아닙니다.  
핵심은 LLM에게 어떤 기준을 주면 검색에 쓸 만한 질문이 나오는지 조정하는 것입니다.

**원 질문**

`반대매매가 발생하기 전에 투자자가 확인해야 할 내용은 무엇인가?`

**작업**

- `practice_viewpoint_prompt`를 직접 작성합니다.
- 원 질문의 주제를 벗어나지 않게 제한합니다.
- 서로 다른 검색 단서를 가진 질문 3개가 나오도록 지시합니다.
- 답변이 아니라 벡터DB 검색에 넣을 질문만 출력하게 합니다.
- 출력 결과가 서로 다른 관점을 갖는지 확인합니다.

**성공 기준**

- 출력된 질문들이 모두 `반대매매`와 투자자 확인 사항을 다룹니다.
- 세 질문의 표현이나 초점이 서로 다릅니다.
- `상황 관점:` 같은 라벨 없이, 검색에 바로 넣을 질문만 출력합니다.
- 문서에 나올 법한 표현을 사용합니다.


In [ ]:
practice_question = "반대매매가 발생하기 전에 투자자가 확인해야 할 내용은 무엇인가?"

practice_viewpoint_prompt = None  # TODO: PromptTemplate.from_template(...)로 관점별 검색 질문 생성 프롬프트를 작성하세요.

if practice_viewpoint_prompt is None:
    print("TODO: practice_viewpoint_prompt를 작성한 뒤 다시 실행하세요.")
else:
    practice_viewpoint_chain = practice_viewpoint_prompt | llm | StrOutputParser()
    practice_raw_queries = practice_viewpoint_chain.invoke({"question": practice_question})  # 모델이 만든 검색 질문 문자열입니다.

    print(practice_raw_queries)


<details>
<summary>정답 코드 보기</summary>

아래 정답은 질문을 세 관점으로 넓히도록 프롬프트를 설계합니다.  
출력 결과가 괜찮다면, 앞의 예제처럼 검색 질문 리스트에 옮겨 검색 결과를 비교할 수 있습니다.

```python
practice_question = "반대매매가 발생하기 전에 투자자가 확인해야 할 내용은 무엇인가?"

practice_viewpoint_prompt = PromptTemplate.from_template("""
금융투자 안내서에서 근거 문서를 찾기 위한 검색 질문을 만드세요.
원 질문의 뜻은 유지하되, 아래 세 관점으로 검색 질문을 1개씩 작성합니다.

[검색 관점]
1. 상황 관점: 어떤 조건에서 문제가 발생하는지 찾는다.
2. 투자자 행동 관점: 투자자가 사전에 확인하거나 해야 할 일을 찾는다.
3. 통지·절차 관점: 금융회사 안내, 통지, 처리 절차와 관련된 근거를 찾는다.

[작성 규칙]
- 원 질문의 주제를 벗어나지 않습니다.
- 답변하지 말고 검색에 넣을 질문만 작성합니다.
- 짧은 구어체보다 안내서 본문에 나올 법한 표현을 사용합니다.

[출력 규칙]
- 설명 문장 없이 검색 질문 3개만 출력합니다.
- 관점명, 번호, 불릿, Markdown 표는 출력하지 않습니다.
- 각 줄은 벡터DB 검색에 바로 넣을 질문 문장만 작성합니다.
- 아래 예시처럼 서로 다른 초점의 질문 3줄만 출력합니다.

예시:
반대매매가 발생하는 조건과 담보부족 상황은 무엇인가?
반대매매 전에 투자자가 추가담보 납부나 계좌 상태에서 확인해야 할 사항은 무엇인가?
금융회사는 반대매매나 담보부족과 관련해 투자자에게 어떤 통지나 절차를 안내하는가?

원 질문: {question}
""")  # 같은 질문을 서로 다른 검색 초점으로 넓히게 합니다.

practice_viewpoint_chain = practice_viewpoint_prompt | llm | StrOutputParser()
practice_raw_queries = practice_viewpoint_chain.invoke({"question": practice_question})  # 모델이 만든 검색 질문 문자열입니다.

print(practice_raw_queries)
```
</details>


지금까지는 **질문 자체를 더 잘 만들어 검색 품질을 높이는 방법**을 살펴봤습니다.  
이제는 질문 대신 **가상의 답변을 만들어 검색하는 HyDE**와, **서로 다른 검색기를 조합하는 Hybrid Retrieval**로 넘어갑니다.



### 2.2 HyDE (Hypothetical Document Embedding)

HyDE는 질문을 바로 검색에 사용하지 않고, LLM을 통해 **가상의 답변(Hypothetical Document)** 을 먼저 만든 뒤,  
그 답변을 임베딩해 문서를 검색하는 기법입니다.

핵심 아이디어는 간단합니다.  
사용자 질문은 짧고 구어체인 경우가 많지만, 실제 문서는 길고 서술형이며 공식 용어를 쓰는 경우가 많습니다.  
HyDE는 이 간극을 줄이기 위해, 질문을 문서에 가까운 말투의 가상 답변으로 한 번 바꿔 검색에 활용합니다.

<img src="image/HyDE.png" width="550">

- 질문: 짧고, 의문문 형태이며, 추상적인 표현을 쓰기 쉬움
- 실제 문서: 길고, 서술형이며, 더 구체적이고 공식적인 표현을 사용함

이 표현 차이 때문에 질문 벡터가 정답 문서 근처에 잘 도달하지 못할 수 있습니다.  
HyDE는 LLM이 정답 문서와 비슷한 말투와 단어를 가진 가상의 답변을 만들게 해서, 이 간극을 줄이려는 접근입니다.


##### 예시: 사내 보안 규정 검색
원본 질문 : "카페에서 와이파이 잡아서 일해도 되나요?"
- 기존 검색의 한계: 실제 규정 문서에는 '카페', '와이파이' 같은 단어가 없을 수 있어 검색에 실패할 확률이 높습니다.

LLM이 생성한 가상 답변 (HyDE):
> "외부 장소에서 업무를 수행할 때는 공용 네트워크 보안 수칙에 따라 반드시 VPN을 연결해야 하며, 화면 보안 필름을 부착해야 합니다.  
인가되지 않은 네트워크 접속은 제한됩니다."

실제 정답 문서 (검색 타겟):  
> [제4조 원격 접속 보안] 사외망 접속 시 인가된 가상 사설망(VPN)을 경유해야 하며,  
개방형 무선 네트워크(Open SSID) 사용은 원칙적으로 금지한다.

##### HyDE의 효과와 한계

**효과:**
- 사용자가 정확한 도메인 용어를 몰라도 검색 성능을 개선할 수 있음
- 질문(Query)과 문서(Document) 간 표현 방식 차이를 LLM이 중간에서 보정
- 실제 문서와 유사한 말투·용어를 가진 가상 답변을 통해 벡터 유사도 상승
- 규정, 매뉴얼, 정책 문서처럼 **질문과 문서 스타일이 크게 다른 경우**에 특히 효과적

**주의사항**
- LLM이 해당 도메인에 대한 이해가 부족한 경우
  - 실제 문서와 무관한 키워드를 생성할 수 있음
  - **이 경우 오히려 검색 성능이 저하될 수 있음**
- 회사 고유 용어, 내부 은어, 프로젝트명 등은
  - HyDE 단독 사용보다 사전 용어 사전, 메타데이터, Hybrid Search와 병행하는 것이 안전함

따라서 HyDE는 **자연어 질문 ↔ 공식 문서 간 간극이 큰 환경에서 선택적으로 사용하는 보조 기법**으로 이해하는 것이 적절합니다.

##### HyDE 기본 예시

**HyDE 가상 답변 생성**

질문을 문서에 가까운 짧은 가상 답변으로 바꾸는 프롬프트와 체인을 만듭니다.


In [ ]:
# 1. 안내서 스타일을 반영한 HyDE 프롬프트 정의
hyde_template = """
사용자의 질문에 대해, 아래 **[문서 성격]**을 바탕으로 짧은 설명문(가상의 답변)을 작성해 주세요.

[문서 성격]
- 금융투자협회의 투자자용 안내서
- 설명은 경고문, 유의사항, 대응 절차처럼 안내서 문체에 가깝게 작성
- 질문에 답하려면 중요할 법한 공식 용어와 행동 지침을 자연스럽게 포함

작성 규칙:
- 4~6줄 안쪽의 짧은 설명문으로 작성
- 질문에 바로 답하는 짧은 안내문처럼 쓸 것
- 문서에 있을 법한 표현과 절차 중심으로 쓸 것
- 질문과 무관한 다른 주제로 크게 확장하지 말 것
- 과도한 배경설명이나 일반 상식은 줄일 것

질문: {query}
가상 답변(문서 스타일 기반):
"""

hyde_prompt = PromptTemplate.from_template(hyde_template)  # 문자열 템플릿을 LangChain 프롬프트 객체로 바꿉니다.

# 2. HyDE용 LLM 정의
hyde_llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 같은 질문에 안정적인 가상 답변을 만들도록 온도를 0으로 둡니다.

# 3. HyDE 체인 구성
hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()  # 질문을 넣으면 가상 답변 문자열이 나오도록 연결합니다.

# 4. 예시 질문 실행
hyde_query = "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?"
hypothetical_doc = hyde_chain.invoke({"query": hyde_query})  # 질문을 문서에 가까운 가상 답변으로 바꿉니다.
print("=== HyDE가 만든 가상 답변 ===")
print(hypothetical_doc)

**HyDE 결과로 문서 검색**

가상 답변을 검색어처럼 사용해 실제 문서에서 가까운 청크를 찾습니다.


In [ ]:
# 5. HyDE로 생성한 가상 문서를 임베딩해 관련 문서를 검색합니다.
results = vectorstore.similarity_search(hypothetical_doc, k=5)  # 가상 문서와 비슷한 실제 문서를 찾습니다.

print("=== [검색 결과] 실제 연결된 문서 내용 ===")
for i, r in enumerate(results[:3], 1):
    page = r.metadata.get("page")  # 원문 PDF의 페이지 정보를 꺼냅니다.
    page_text = f" / page {page + 1}" if isinstance(page, int) else ""  # 사람이 보는 페이지 번호로 바꿉니다.
    print(f"\n[문서 {i}{page_text}]")
    print(r.page_content[:300])  # 본문 앞부분만 확인해 연결 방향을 봅니다.


##### 실습 문제 2: HyDE 검색용 문서 만들기

HyDE는 검색 결과를 먼저 보고 요약하는 방식이 아닙니다.  
사용자 질문만 보고, 실제 문서에 가까운 말투의 **가상 문서**를 만든 뒤 그 문장으로 검색합니다.

좋은 HyDE 문서는 실제 문서와 말투, 표현, 용어가 비슷해야 합니다.  
먼저 원본 PDF를 열고 `모바일폰 사용 시 유의사항`, `모바일폰 금융거래 10계명` 부분을 확인합니다.

단, 원문을 프롬프트에 그대로 붙여 넣지는 않습니다.  
문서의 **표현 방식**을 따라 하는 것이 목표입니다.

확인할 파일: `data/금융투자협회_투자길라잡이_2018.pdf`

**작업**

- `practice_hyde_prompt`를 직접 작성합니다.
- 원본 PDF에서 반복되는 표현과 문체를 관찰합니다.
- 질문에 바로 답하는 짧은 챗봇 답변이 아니라, 안내서에 들어 있을 법한 문단을 만들게 합니다.
- `모바일폰 금융거래`, `금융거래 비밀번호`, `OTP`, `잠금기능`, `잠금비밀번호`, `무선랜(Wi-Fi)`처럼 원문에서 확인한 표현을 자연스럽게 쓰게 합니다.
- 질문에 없는 세부 사실이나 절차를 과하게 지어내지 않도록 제한합니다.
- 원 질문 검색 결과와 HyDE 검색 결과를 비교합니다.


In [ ]:
practice_hyde_query = "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?"

# 먼저 data/금융투자협회_투자길라잡이_2018.pdf에서
# '모바일폰 사용 시 유의사항'과 '모바일폰 금융거래 10계명' 부분을 확인합니다.
# 원문의 표현과 문체를 관찰한 뒤 아래 프롬프트를 작성하세요.

practice_hyde_prompt = None  # TODO: PromptTemplate.from_template(...)로 검색용 가상 문서 프롬프트를 작성하세요.

if practice_hyde_prompt is None:
    print("\nTODO: practice_hyde_prompt를 작성한 뒤 HyDE 검색 결과를 비교하세요.")
else:
    practice_hyde_chain = practice_hyde_prompt | llm | StrOutputParser()
    practice_hypothetical_doc = practice_hyde_chain.invoke({"query": practice_hyde_query})  # 질문만 넣어 검색용 가상 문서를 만듭니다.

    print("\n=== 실습 문제 2 - HyDE 검색용 가상 문서 ===")
    print(practice_hypothetical_doc)

    direct_docs = retriever.invoke(practice_hyde_query)  # 원 질문 그대로 검색한 결과입니다.
    practice_hyde_results = vectorstore.similarity_search(practice_hypothetical_doc, k=3)  # HyDE 문서로 검색한 결과입니다.

    preview_docs("실습 문제 2 - 원 질문 검색 결과", direct_docs, max_items=2)
    preview_docs("실습 문제 2 - HyDE 검색 결과", practice_hyde_results, max_items=3)


<details>
<summary>정답 코드 보기</summary>

아래 정답은 원본 PDF에서 확인한 말투와 용어를 프롬프트 규칙으로 옮긴 예시입니다.  
원문을 그대로 넣는 것이 아니라, 안내서 문체와 용어를 따라 하도록 지시합니다.

```python
practice_hyde_prompt = PromptTemplate.from_template("""
사용자의 질문을 바탕으로 벡터DB 검색에 사용할 가상 문서를 작성하세요.

[문서 성격]
- 금융투자협회의 투자자 안내서
- 투자자가 지켜야 할 사항을 항목별로 안내하는 문서
- 구어체 답변보다 공식적인 유의사항 문체
- '~해야 합니다', '~하지 않습니다', '~에 유의해야 합니다'처럼 행동 기준을 설명하는 문체

[원본 PDF에서 관찰한 표현]
- 모바일폰 금융거래
- 금융거래 비밀번호
- 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP)
- 모바일폰 사용환경, 모바일폰 보안업데이트
- 잠금기능, 잠금비밀번호
- 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi)

[작성 규칙]
1. 질문에 바로 답하는 챗봇 말투가 아니라, 안내서 본문처럼 작성합니다.
2. 위 표현 중 질문과 자연스럽게 맞는 표현을 포함합니다.
3. 원문처럼 행동 기준을 나열하는 유의사항 문체로 씁니다.
4. 질문에 없는 신고처, 보상 기준, 앱 설치 방법은 새로 만들지 않습니다.
5. 5~7줄의 짧은 검색용 문서로 작성합니다.

질문: {query}
검색용 가상 문서:
""")  # 질문을 안내서 문체의 검색용 문단으로 바꿉니다.

practice_hyde_chain = practice_hyde_prompt | llm | StrOutputParser()
practice_hypothetical_doc = practice_hyde_chain.invoke({"query": practice_hyde_query})  # 질문만 넣어 HyDE 문서를 만듭니다.

print("\n=== 실습 문제 2 - HyDE 검색용 가상 문서 ===")
print(practice_hypothetical_doc)

direct_docs = retriever.invoke(practice_hyde_query)  # 원 질문으로 검색합니다.
practice_hyde_results = vectorstore.similarity_search(practice_hypothetical_doc, k=3)  # HyDE 문서로 검색합니다.

preview_docs("실습 문제 2 - 원 질문 검색 결과", direct_docs, max_items=2)
preview_docs("실습 문제 2 - HyDE 검색 결과", practice_hyde_results, max_items=3)
```
</details>


#### HyDE 사용 시 주의할 점

방금 예제처럼 HyDE가 관련 문서를 잘 끌어올릴 때도 있지만,
항상 원하는 방향으로 동작하는 것은 아닙니다.

가상 답변이 문서의 실제 표현과 어긋나면 검색 결과도 빗나갈 수 있습니다.

**왜 빗나갈 수 있나요?**

1. **문맥 불일치 (Context Mismatch)**  
   - **LLM의 상상:** "모바일 투자라면 앱 사용성, 편의성, 생산성 팁을 말하겠구나"
   - **실제 데이터:** "모바일폰 사용 시 유의사항", "비밀번호", "공인인증서", "보안", "금융사고"처럼 더 구체적이고 규정형인 표현이 등장

2. **키워드 오염 (Keyword Pollution)**  
   - LLM이 만든 가상 답변에 일반적인 디지털 금융 키워드가 너무 많으면,
     문서의 핵심 표현보다 일반적인 금융 문장이 우선 검색될 수 있습니다.

3. **가상 문서 할루시네이션**  
   - HyDE의 가상 문서는 최종 답변이 아니라 검색에만 사용되므로, 할루시네이션이 곧바로 사용자 답변으로 나가지는 않습니다.
   - 그래도 원문에 없는 기관명, 절차, 수치, 보상 기준 같은 내용이 들어가면 그 표현과 가까운 엉뚱한 문서가 검색될 수 있습니다.
   - 따라서 가상 문서는 답변 근거로 쓰지 말고, 최종 답변은 반드시 검색된 실제 문서 본문을 근거로 생성해야 합니다.

**교훈:**  
"검색에만 쓰니까 할루시네이션이 생겨도 괜찮다"가 아니라,
"검색 방향을 바꾸는 입력이므로 과도한 상상을 줄여야 한다"에 가깝습니다.  
HyDE는 금융 안내서처럼 **정확한 용어와 절차**가 중요한 문서에서는 보조 기법으로만 쓰고,
문서의 톤과 공식 용어를 최대한 반영해야 합니다.
        

### Semantic Routing: 검색할 질문만 벡터DB로 보내기

Semantic Routing은 질문의 의미를 보고 다음 처리 경로를 고르는 접근입니다.  
여기서는 그중 RAG 시스템에서 자주 쓰이는 형태인 **내 문서로 답할 질문만 벡터DB로 보내는 라우팅**을 다룹니다.

RAG 시스템을 운영할 때 가장 자주 생기는 낭비는 **문서로 답할 수 없는 질문까지 벡터DB에 검색하는 것**입니다.

예를 들어 금융투자협회 투자자 안내서 기반 챗봇이라면 아래 질문은 RAG가 잘 맞습니다.

- 모바일 금융거래에서 OTP와 비밀번호를 어떻게 관리해야 하는가?
- 반대매매가 실행되기 전 추가담보 납부와 통지는 어떻게 되는가?
- 임의매매 분쟁에서 손해배상 책임은 어떻게 판단하는가?

반대로 아래 질문은 안내서 벡터DB를 검색해도 좋은 근거가 나오기 어렵습니다.

- 오늘 코스피 지수는 어떻게 움직이고 있나요?
- 안녕하세요. 사용법을 알려주세요.
- 도쿄일렉트론코리아의 최신 채용 공고를 알려주세요.

이런 질문은 벡터DB 검색을 생략하고, 일반 답변이나 다른 도구로 넘기는 편이 낫습니다.


예전에는 질문 예시를 임베딩하고 라우트별 대표 벡터와 코사인 유사도를 비교하는 방식으로 Semantic Routing을 설명하는 경우가 많았습니다.
작은 데모로는 이해하기 쉽지만, 실무의 RAG 여부 판단에는 한계가 큽니다.

- 라우트 밖 질문도 반드시 어느 한 라우트로 들어갑니다.
- 최신 정보, 인사말, 문서 범위 밖 질문을 안정적으로 걸러내기 어렵습니다.
- 질문이 복합적일 때 검색할 부분과 검색하지 않을 부분을 나누기 어렵습니다.

여기서는 라우트를 `rag`와 `direct` 두 가지로 단순화합니다.
핵심은 **내가 가진 문서로 답할 질문만 검색하고, 나머지는 벡터DB를 건너뛴다**는 것입니다.

| 라우트 | 처리 방식 | 예시 |
|---|---|---|
| `rag` | 벡터DB에서 관련 문서를 찾고, 근거 기반 답변 생성 | 안내서에 들어 있을 법한 투자자 보호, 금융거래 유의사항, 분쟁 사례 질문 |
| `direct` | 벡터DB 검색 생략 | 인사말, 사용법 질문, 최신 시세·뉴스, 문서 범위 밖 질문 |

흐름은 아래처럼 단순합니다.

```text
사용자 질문
  -> route = rag    -> 벡터DB 검색 -> 근거 기반 답변
  -> route = direct -> 벡터DB 검색 생략 -> 일반 안내 또는 다른 도구 후보
```

여기서는 검색용 문장을 따로 만들지 않고, `rag`로 판단된 질문을 그대로 검색합니다.  
검색어를 다시 쓰는 단계는 Query Rewriting이나 Multi-Query가 필요할 때 별도로 붙입니다.
실무에서 이 흐름을 함수 `if` 문으로만 관리하면 분기가 늘어날수록 코드가 복잡해집니다.
LangGraph를 쓰면 `라우터 노드 -> RAG 노드 또는 직접 답변 노드`처럼 노드와 조건부 엣지로 나누어 관리할 수 있어, 분기·재시도·로그 기록을 더 깔끔하게 붙일 수 있습니다.

이 판단은 규칙이나 키워드로도 시작할 수 있지만, 질문 의도와 문서 범위를 함께 봐야 할 때는 LLM의 구조화 출력을 쓰는 편이 더 현실적입니다.


**LLM으로 RAG 여부 판단하기**

LLM이 자유문장으로 답하면 후속 코드가 처리하기 어렵습니다.
그래서 `route`, `reason`이 들어 있는 JSON 형식으로 답하게 만듭니다.
라우터는 검색 여부만 판단하고, 검색어 재작성은 하지 않습니다.

운영 코드에서는 구조화 출력 기능으로 더 엄격하게 검증할 수 있습니다.
여기서는 초심자 예제 흐름을 단순하게 보기 위해 표준 `json` 모듈로 파싱합니다.


In [ ]:
import json  # LLM이 출력한 JSON 문자열을 파이썬 딕셔너리로 바꿀 때 씁니다.
from langchain_core.prompts import ChatPromptTemplate  # 시스템 메시지와 사용자 질문을 하나의 프롬프트로 묶습니다.

# 라우터에게 줄 판단 기준입니다.
# 모델이 마음대로 판단하지 않도록, 어떤 질문을 RAG로 보낼지와 어떤 질문을 건너뛸지 먼저 정합니다.
routing_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융투자협회 투자자 안내서 기반 RAG 챗봇의 라우터입니다.
사용자 질문을 보고 벡터DB 검색이 필요한지 판단하세요.

[rag로 보낼 질문]
- 금융투자협회 투자자 안내서에 들어 있을 법한 내용
- 투자자 보호, 금융상품 설명, 반대매매, 금융사기 예방, 모바일 금융거래 보안, 분쟁 사례, 절차와 유의사항
- 문서 근거를 확인해야 정확히 답할 수 있는 질문

[direct로 보낼 질문]
- 인사말, 챗봇 사용법처럼 문서 검색이 필요 없는 질문
- 오늘/최신/현재 시세, 뉴스, 공고처럼 외부 최신 정보가 필요한 질문
- 안내서 범위를 벗어난 회사·제품·개인·일반 상식 질문

출력 규칙:
- 설명 문장을 덧붙이지 말고 JSON만 출력합니다.
- route는 반드시 "rag" 또는 "direct" 중 하나입니다.
- 확실하지 않아도 안내서에 근거가 있을 법하면 rag를 선택합니다.

출력 형식:
{{
  "route": "rag 또는 direct",
  "reason": "왜 이 라우트를 골랐는지 한 문장"
}}
""",
    ),
    ("human", "질문: {query}"),  # 실제 사용자 질문은 {query} 자리에 들어갑니다.
])

# 프롬프트 -> LLM -> 문자열 출력 순서로 라우터 체인을 만듭니다.
# 결과는 아직 문자열이므로 decide_rag_route() 안에서 json.loads()로 바꿉니다.
rag_router = routing_prompt | llm | StrOutputParser()

def decide_rag_route(query: str) -> dict:
    """질문 하나를 받아 RAG 검색이 필요한지 판단합니다."""
    raw_output = rag_router.invoke({"query": query})  # LLM은 JSON 모양의 문자열을 반환합니다.
    decision = json.loads(raw_output)  # 문자열을 {'route': ..., 'reason': ...} 딕셔너리로 바꿉니다.

    # 혹시 모델이 예상 밖 값을 냈을 때 예제 코드가 이상한 경로로 흐르지 않게 최소한의 보정을 합니다.
    if decision.get("route") not in ["rag", "direct"]:
        decision["route"] = "direct"
        decision["reason"] = "라우팅 결과가 명확하지 않아 검색을 생략합니다."

    return decision

# 서로 다른 유형의 질문을 넣어 라우터가 어떻게 판단하는지 먼저 확인합니다.
sample_questions = [
    "모바일 금융거래에서 OTP와 비밀번호를 어떻게 관리해야 하나요?",  # 안내서 근거가 필요한 질문입니다.
    "오늘 코스피 지수는 어떻게 움직이고 있나요?",  # 최신 시세라서 안내서 검색이 맞지 않습니다.
    "안녕하세요. 사용법을 알려주세요.",  # 인사말/사용법은 벡터DB 검색 없이도 답할 수 있습니다.
]

for query in sample_questions:
    decision = decide_rag_route(query)  # 질문마다 rag/direct 판단 결과를 받습니다.
    print("질문:", query)
    print("route:", decision["route"])  # 실제 분기 조건으로 쓰는 값입니다.
    print("reason:", decision["reason"])  # 왜 그렇게 판단했는지 확인합니다.
    print("---")


**라우팅 결과로 RAG 실행 또는 검색 생략**

`rag`로 판단된 질문만 벡터DB를 검색합니다.
`direct`로 판단된 질문은 검색 단계를 건너뛰고, 문서 검색을 쓰지 않는 답변 체인으로 보냅니다.


In [ ]:
def format_context(docs):
    """검색된 문서 리스트를 LLM 프롬프트에 넣기 좋은 하나의 문자열로 바꿉니다."""
    chunks = []  # 문서마다 보기 좋은 문자열을 만들어 담을 리스트입니다.
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page")  # PDF에서 몇 쪽 문서였는지 메타데이터에서 꺼냅니다.
        page_text = f"p.{page + 1}" if isinstance(page, int) else "page unknown"  # pypdf의 page는 0부터 시작하므로 1을 더합니다.
        chunks.append(f"[{i}] {page_text}\n{doc.page_content}")  # 번호, 페이지, 본문을 함께 붙입니다.
    return "\n\n".join(chunks)  # 여러 문서를 빈 줄로 구분해 하나의 context로 만듭니다.

# RAG 경로에서 사용할 답변 프롬프트입니다.
# 검색된 근거 문서만 보고 답하도록 제한해 문서 밖 내용을 섞지 않게 합니다.
rag_answer_prompt = ChatPromptTemplate.from_template("""
당신은 금융투자협회 투자자 안내서를 근거로 답하는 RAG 챗봇입니다.
아래 [근거 문서]에 있는 내용만 바탕으로 답하세요.
근거가 부족하면 문서에서 확인되는 범위까지만 말하세요.

질문: {query}

[근거 문서]
{context}

답변:
""")

# direct 경로에서 사용할 답변 프롬프트입니다.
# 이 경로는 벡터DB를 검색하지 않으므로 짧은 안내만 합니다.
direct_answer_prompt = ChatPromptTemplate.from_template("""
벡터DB 검색 없이 짧게 답하세요.

질문: {query}

답변 규칙:
- 인사말이나 사용법 질문이면 짧게 안내합니다.
- 최신 시세, 뉴스, 채용 공고처럼 현재 정보가 필요한 질문이면 실시간 자료가 필요하다고 말합니다.
- 답하기 어려운 질문이면 확인할 수 없다고 말합니다.

답변:
""")

# 프롬프트 -> LLM -> 문자열 출력 순서로 두 개의 답변 체인을 만듭니다.
rag_answer_chain = rag_answer_prompt | llm | StrOutputParser()
direct_answer_chain = direct_answer_prompt | llm | StrOutputParser()

def answer_with_rag_gate(query: str, k: int = 4):
    """질문을 라우팅한 뒤, 필요한 경우에만 벡터DB를 검색해 답변합니다."""
    decision = decide_rag_route(query)  # 1단계: 질문이 rag인지 direct인지 먼저 판단합니다.
    print("route:", decision["route"])
    print("reason:", decision["reason"])

    if decision["route"] == "direct":  # 2-A단계: 검색이 필요 없으면 벡터DB를 호출하지 않습니다.
        answer = direct_answer_chain.invoke({"query": query})
        return decision, [], answer  # 검색 문서가 없으므로 docs 자리에 빈 리스트를 돌려줍니다.

    # 2-B단계: rag로 판단된 질문만 벡터DB 검색을 실행합니다.
    docs = retriever.invoke(query)[:k]  # 원 질문 그대로 검색해 상위 k개 문서를 가져옵니다.
    context = format_context(docs)  # 검색 결과를 프롬프트에 넣을 문자열로 정리합니다.
    answer = rag_answer_chain.invoke({"query": query, "context": context})  # 근거 문서를 바탕으로 답변을 생성합니다.
    return decision, docs, answer

# 예시 질문 하나로 전체 흐름을 실행합니다.
query = "모바일 금융거래에서 OTP와 비밀번호를 어떻게 관리해야 하나요?"
decision, docs, answer = answer_with_rag_gate(query)

print("검색 문서 수:", len(docs))  # rag면 문서 수가 나오고, direct면 0이 나옵니다.
print("\n답변:")
print(answer)

##### 확인 예제: RAG gate 테스트하기

아래 질문들을 넣어 보고, 어떤 질문에서 벡터DB 검색이 실행되는지 확인합니다.
검색 결과가 없어도 되는 질문까지 무조건 벡터DB에 넣는 것보다, 먼저 문서 범위를 판단하는 편이 운영 비용과 답변 품질 관리에 유리합니다.


In [ ]:
practice_route_queries = [
    "투자자 유형은 어떤 기준으로 나누나요?",  # 안내서 안에서 근거를 찾아야 하는 질문입니다.
    "보이스피싱을 당했을 때 어떤 절차를 밟아야 하나요?",  # 금융사기 대응 절차라 RAG 후보입니다.
    "오늘 코스피 지수는 어떻게 움직이고 있나요?",  # 최신 시세라 안내서 벡터DB 검색 대상이 아닙니다.
    "안녕하세요. 사용법을 알려주세요.",  # 인사말/사용법은 검색 없이 짧게 답할 수 있습니다.
    "도쿄일렉트론코리아의 최신 채용 공고를 알려주세요.",  # 안내서 범위 밖이며 최신 정보가 필요합니다.
]

for query in practice_route_queries:
    decision = decide_rag_route(query)  # 각 질문을 rag/direct 중 하나로 분류합니다.
    print("질문:", query)
    print("route:", decision["route"])
    print("reason:", decision["reason"])

    if decision["route"] == "rag":
        # rag로 판단된 질문만 실제 벡터DB 검색을 실행합니다.
        search_docs = retriever.invoke(query)[:2]
        print("벡터DB 검색 실행:", len(search_docs), "건")
        if search_docs:
            # 결과 전체를 출력하면 너무 길어지므로 첫 문서 앞부분만 확인합니다.
            print("첫 결과:", search_docs[0].page_content[:120].replace("\n", " "))
    else:
        # direct로 판단되면 retriever.invoke(...) 자체를 호출하지 않습니다.
        print("벡터DB 검색 생략")
    print("---")


핵심은 Semantic Routing을 단순 라벨 분류로 외우는 것이 아니라, **검색이 필요한 질문인지 먼저 판단하는 구조**로 이해하는 것입니다.

- 가진 문서로 답할 수 있으면 RAG를 실행합니다.
- 문서 범위 밖이거나 최신 정보가 필요하면 벡터DB 검색을 생략합니다.
- 운영 환경에서는 이 판단 결과를 로그로 남기고, 잘못 라우팅된 질문을 평가셋에 추가합니다.

이 관점 위에 Hybrid Retrieval을 얹으면, `rag`로 들어온 질문 안에서 Dense, BM25, RRF 같은 검색 전략을 더 정교하게 비교할 수 있습니다.


### 2.3 Hybrid Retrieval

**Hybrid Retrieval**은 서로 다른 검색 방식의 약점을 서로 보완해, 정답 후보를 더 넓고 안정적으로 모으는 방식입니다.

| 방식 | 장점 | 단점 |
|---|---|---|
| 벡터 기반 의미 검색 | 표현이 달라도 의미가 비슷하면 잘 잡아냄 | 정확한 용어, 숫자, 법령 표기 검색에 약할 수 있음 |
| 키워드 기반 매칭 검색(BM25) | 용어 일치, 법령 구문, 코드, 표기 검색에 강함 | 표현이 조금만 달라도 쉽게 빗나갈 수 있음 |
| Hybrid | Dense가 놓친 정확한 용어 일치를 Sparse가 보완하고, Sparse가 놓친 의미 유사 표현을 Dense가 보완함 | 시스템 복잡성이 증가함 |

##### Hybrid 결과 병합 방식

Hybrid Retrieval에서는 Dense 검색과 BM25 검색을 각각 수행한 뒤, 두 결과를 어떤 방식으로 합칠지 결정해야 합니다.
아래 표는 검색 결과를 병합할 때 자주 쓰는 대표적인 방법들입니다.

| 방식 | 설명 | 특징 |
|---|---|---|
| **Parallel Retrieval** | Dense 검색과 BM25 검색을 각각 수행한 뒤 결과를 단순 병합 | 구현이 단순하지만 최종 순위가 불안정할 수 있음 |
| **Weighted Score Fusion** | 두 검색 방식의 점수를 정규화한 뒤 가중 평균으로 결합 | 가중치(alpha, beta)를 수동으로 조정해야 함 |
| **Reciprocal Rank Fusion (RRF)** | 각 검색 결과의 순위를 기반으로 점수를 계산해 결합 | 가중치 튜닝 없이도 안정적인 성능 |

아래에서는 **RRF 기반 Hybrid Retrieval**을 중심으로 봅니다.



##### Hybrid Retrieval 기본 예시



**Dense/BM25 검색기 준비**

같은 청크를 의미 검색과 키워드 검색 두 방식으로 찾을 수 있게 준비합니다.


In [ ]:
from langchain_community.retrievers import BM25Retriever  # 키워드 기반 BM25 검색기를 만듭니다.

# Dense와 Sparse는 같은 split_docs 위에서 비교해야 Hybrid 결합 결과를 해석하기 쉽습니다.
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 임베딩 기반으로 상위 6개 청크를 찾습니다.
bm25_retriever = BM25Retriever.from_documents(split_docs)  # 같은 청크를 키워드 검색용 색인으로도 만듭니다.
bm25_retriever.k = 6  # BM25도 상위 6개까지 가져오게 맞춥니다.

# Hybrid 데모는 Dense와 BM25 결과가 일부 겹쳐 RRF 누적 효과가 잘 보이는 질문으로 통일합니다.
hybrid_demo_query = "반대매매가 실행되기 전 추가담보 납부와 통지는 어떻게 되는가?"

print("Dense/BM25 공통 청크 수:", len(split_docs))
print("Hybrid 비교 질문:", hybrid_demo_query)

#### Parallel Hybrid Retrieval (기본 조합)

Dense 검색과 Sparse 검색을 병렬로 수행한 뒤 결과를 단순히 이어 붙여 봅니다.



**Dense와 BM25 결과 비교**

같은 질문을 두 검색기에 넣고 어떤 청크가 먼저 나오는지 확인합니다.


In [ ]:
# 실제로 테스트할 질문을 정합니다.
query = hybrid_demo_query  # Dense와 Sparse를 같은 질문으로 비교합니다.

dense_docs = dense_retriever.invoke(query)  # 의미 기반 검색 결과입니다.
sparse_docs = bm25_retriever.invoke(query)  # 키워드 기반 검색 결과입니다.

print(f"=== [Dense 검색 결과: {len(dense_docs)}개] ===")
for i, doc in enumerate(dense_docs[:4], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:140])  # 본문 앞부분만 봐도 검색 방향을 비교할 수 있습니다.
    print("---")

print(f"\n=== [Sparse(BM25) 검색 결과: {len(sparse_docs)}개] ===")
for i, doc in enumerate(sparse_docs[:4], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:140])  # 본문 앞부분만 봐도 검색 방향을 비교할 수 있습니다.
    print("---")

**단순 병합 결과 확인**

두 검색 결과를 그냥 이어 붙였을 때 중복과 순위 문제가 생기는지 봅니다.


In [ ]:
# Dense 결과와 Sparse 결과를 그대로 이어 붙인 예시입니다.
# 이렇게 단순 합치면 중복이 생길 수 있고, 두 검색기의 순위도 그대로 섞여 들어갑니다.
combined_docs = dense_docs + sparse_docs

print(f"\n=== [Combined 결과: {len(combined_docs)}개] ===")
for i, doc in enumerate(combined_docs[:8], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:110])
    print("---")

→ 단순 병합은 구현은 쉽지만, 결과에 **중복이 생길 수 있고** 두 검색기의 순위도 그대로 섞여 들어갑니다.  
즉, 같은 문서를 한 번만 남기고, 여러 검색기에서 **꾸준히 상위에 나온 문서**를 더 앞으로 올리는 정리가 필요합니다.


#### Reciprocal Rank Fusion (RRF)

RRF는 서로 다른 검색 결과의 **순위 정보만** 사용해 결과를 다시 합치는 방식입니다.

```
score = 1 / (k + rank + 1)
```

- `rank`: 각 검색 결과 안에서의 순위
- `k`: 순위 영향을 완만하게 만드는 상수. 보통 60을 많이 사용

핵심은 **한 검색기에서만 우연히 높게 나온 문서보다, 여러 검색기에서 꾸준히 상위에 등장한 문서**를 더 앞에 두는 것입니다.  
즉, Dense와 Sparse 둘 다에서 순위가 높은 문서가 위로 올라오도록 정리하는 방식이라고 이해하면 됩니다.


**RRF로 검색 결과 합치기**

Dense와 BM25의 순위를 점수로 바꿔 문서별 최종 순서를 계산합니다.


In [ ]:
from langchain_core.retrievers import BaseRetriever  # 커스텀 검색기를 LangChain 표준 Retriever처럼 쓰기 위해 상속합니다.

# 각 검색기의 순위만 사용해 문서별 누적 점수를 계산합니다.
def reciprocal_rank_fusion_with_scores(result_lists, k=60):  # 여러 검색기의 순위를 하나의 점수표로 합칩니다.
    """
    Reciprocal Rank Fusion(RRF)을 직접 구현한 함수

    result_lists : List[List[Document]]
        Dense, Sparse 등 여러 retriever가 반환한 검색 결과 리스트들의 묶음
        예: [dense_results, bm25_results]

    k : int
        순위 점수 차이를 너무 급격하게 만들지 않도록 완충하는 상수
        보통 60을 많이 사용합니다.

    반환값 : List[dict]
        {"doc": Document, "score": 누적점수} 형태의 리스트를 점수순으로 정렬해 돌려줍니다.
    """
    scores = {}  # 문서별 누적 점수를 저장합니다.

    for results in result_lists:  # Dense, Sparse처럼 각 검색기 결과를 차례로 봅니다.
        for rank, doc in enumerate(results):
            score = 1.0 / (k + rank + 1)  # 순위가 높을수록 더 큰 점수를 받습니다.
            doc_id = doc.metadata.get("id", doc.page_content)  # 같은 문서인지 비교할 기준 ID입니다.

            if doc_id not in scores:  # 처음 본 문서면 점수표에 등록합니다.
                scores[doc_id] = {"doc": doc, "score": 0.0}
            scores[doc_id]["score"] += score  # Dense 점수와 Sparse 점수를 문서별로 누적합니다.

    return sorted(scores.values(), key=lambda item: item["score"], reverse=True)

# 최종적으로는 점수순으로 정렬된 문서 리스트만 돌려줍니다.
def reciprocal_rank_fusion(result_lists, k=60):  # 점수표에서 문서 리스트만 꺼냅니다.
    """RRF 점수표에서 문서 객체만 꺼내 최종 순위 리스트로 반환합니다."""
    ranked = reciprocal_rank_fusion_with_scores(result_lists, k=k)  # 문서와 점수를 함께 계산합니다.
    return [item["doc"] for item in ranked]

# 여러 retriever를 한 번에 호출해 RRF로 합친 결과를 만듭니다.
def hybrid_rrf_retrieve(query, retrievers, k=60):
    """동일한 질문을 여러 retriever에 넣고, 결과를 RRF로 결합합니다."""
    results = [r.invoke(query) for r in retrievers]  # 같은 질문을 각 검색기에 넣습니다.
    return reciprocal_rank_fusion(results, k=k)

# LangChain 체인 안에서도 일반 retriever처럼 쓰기 위한 래퍼입니다.
class HybridRRFRetriever(BaseRetriever):
    retrievers: list  # 함께 결합할 검색기 목록입니다.
    k: int = 60  # RRF 점수 계산에 쓰는 완충 상수입니다.

    def _get_relevant_documents(self, query: str, *, run_manager=None):  # 일반 invoke에서 호출됩니다.
        return hybrid_rrf_retrieve(query, self.retrievers, k=self.k)

    async def _aget_relevant_documents(self, query: str, *, run_manager=None):  # 비동기 호출에서도 같은 로직을 씁니다.
        return hybrid_rrf_retrieve(query, self.retrievers, k=self.k)

# 출력에서 Dense 순위와 Sparse 순위를 함께 보기 위한 보조 맵입니다.
def build_rank_map(docs):  # 문서 ID를 보면 원래 몇 위였는지 바로 찾게 해 줍니다.
    return {doc.metadata.get("id", doc.page_content): rank + 1 for rank, doc in enumerate(docs)}

# 개별 검색 결과 확인 (비교용)
query = hybrid_demo_query

dense_results = dense_retriever.invoke(query)  # Dense 검색기의 원래 순위입니다.
bm25_results = bm25_retriever.invoke(query)  # Sparse 검색기의 원래 순위입니다.

# Hybrid retriever는 Reranking과 Post-Retrieval 예제에서도 그대로 재사용합니다.
hybrid_rrf = HybridRRFRetriever(
    retrievers=[dense_retriever, bm25_retriever],  # Dense와 BM25 결과를 함께 결합합니다.
    k=60,  # 순위 차이를 완만하게 반영하는 RRF 기본값입니다.
)

# 같은 질문에 대해 RRF 점수표를 먼저 만들고, 그 순서대로 최종 문서를 봅니다.
rrf_ranked = reciprocal_rank_fusion_with_scores([dense_results, bm25_results], k=60)  # RRF 점수표를 만듭니다.
hybrid_results = [item["doc"] for item in rrf_ranked]  # 최종 문서 순위만 따로 꺼냅니다.

dense_rank_map = build_rank_map(dense_results)  # 문서별 Dense 순위를 빠르게 찾습니다.
sparse_rank_map = build_rank_map(bm25_results)  # 문서별 Sparse 순위를 빠르게 찾습니다.

print("=== RRF 상위 문서의 순위 비교 ===")
print("(참고: '-'는 해당 검색기의 top-k 결과에는 없었다는 뜻입니다.)")
for item in rrf_ranked[:5]:
    doc = item["doc"]
    doc_id = doc.metadata.get("id", doc.page_content)
    rrf_score = item["score"]
    print(
        f"id={doc_id} | dense_rank={dense_rank_map.get(doc_id, '-')} | "
        f"sparse_rank={sparse_rank_map.get(doc_id, '-')} | rrf_score={rrf_score:.4f}"
    )

print("\n=== RRF 상위 문서 본문 일부 ===")
for doc in hybrid_results[:3]:
    page = doc.metadata.get("page")
    page_text = f", page={page + 1}" if isinstance(page, int) else ""
    print("---")
    print(f"[id={doc.metadata.get('id')}{page_text}]")
    print(doc.page_content[:220])  # 본문 앞부분만 출력해 어떤 문서인지 확인합니다.


##### RRF의 장점
- 점수 스케일을 맞추지 않아도 됨
- Dense / Sparse 결과를 함께 쓸 때 안정적임
- 구현이 단순하면서도 실무에서 자주 사용됨



#### Hybrid Retrieval 결과 비교 실험

아래에서는 `단순 병합`과 달리, RRF가 **Dense와 Sparse의 순위를 문서별로 누적해 다시 정렬**하는 모습을 봅니다.  
같은 문서가 양쪽 검색 결과에 함께 나오면 점수가 더해져 위로 올라가기 쉽습니다.
문서별 `dense_rank`, `sparse_rank`, `rrf_score`를 함께 보면서 결과를 해석합니다.


**Dense, BM25, Hybrid 비교**

세 방식의 상위 결과를 나란히 출력해 Hybrid가 후보를 어떻게 정리하는지 확인합니다.


In [ ]:
def preview_results(label, docs):  # 검색 결과 상위 2개를 짧게 보여 줍니다.
    print(f"\n=== {label} (Top 2) ===")
    for d in docs[:2]:
        print("---")
        print(d.page_content[:200])  # 본문 앞부분만 출력해 비교합니다.

# 실제로 테스트할 질문을 정합니다.
query = hybrid_demo_query  # 같은 질문으로 Dense, Sparse, Hybrid를 나란히 비교합니다.
#query = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"
#query = "부당권유란 무엇이며 어떤 경우 위법성이 문제되는가?"

dense_res = dense_retriever.invoke(query)  # Dense 검색 결과입니다.
sparse_res = bm25_retriever.invoke(query)  # BM25 검색 결과입니다.
hybrid_res = hybrid_rrf.invoke(query)  # 두 결과를 합친 Hybrid 검색 결과입니다.

preview_results("Dense Only", dense_res)
preview_results("Sparse Only", sparse_res)
preview_results("Hybrid (RRF, 양쪽 순위 누적)", hybrid_res)

##### 실습 문제 3: Dense, BM25, Hybrid 결과 비교하기

Hybrid Retrieval은 좋은 도메인 질문을 만드는 문제가 아닙니다.  
같은 질문을 여러 검색기에 넣었을 때 결과가 어떻게 달라지는지 확인하는 것이 핵심입니다.

질문은 이미 준비되어 있습니다.

- 정확한 용어 중심 질문
- 사용자가 말할 법한 풀어쓴 표현 중심 질문

할 일은 세 검색기를 같은 질문에 적용한 뒤, Top 결과의 ID를 비교하는 것입니다.  
Dense와 BM25가 공통으로 찾은 문서, 한쪽에서만 찾은 문서, Hybrid 상위에 올라온 문서를 함께 봅니다.


In [ ]:
comparison_queries = {
    "정확한 용어 중심 질문": "반대매매와 추가담보 납부 통지에 대해 투자자가 확인해야 할 사항은 무엇인가?",
    "풀어쓴 표현 중심 질문": "담보가 부족해서 주식이 자동으로 팔리기 전에 고객이 뭘 확인해야 하나요?",
}
top_k = 4

for label, query in comparison_queries.items():
    print(f"\n===== {label} =====")
    print("질문:", query)

    dense_docs = None  # TODO: dense_retriever로 query를 검색하세요.
    sparse_docs = None  # TODO: bm25_retriever로 query를 검색하세요.
    hybrid_docs = None  # TODO: hybrid_rrf로 query를 검색하세요.

    if dense_docs is None or sparse_docs is None or hybrid_docs is None:
        print("TODO: dense_docs, sparse_docs, hybrid_docs를 완성하세요.")
        break

    dense_ids = [doc.metadata.get("id") for doc in dense_docs[:top_k]]
    sparse_ids = [doc.metadata.get("id") for doc in sparse_docs[:top_k]]
    hybrid_ids = [doc.metadata.get("id") for doc in hybrid_docs[:top_k]]

    shared_ids = None  # TODO: Dense와 BM25 양쪽 Top 결과에 모두 들어간 id를 구하세요.
    dense_only_ids = None  # TODO: Dense Top 결과에만 있는 id를 구하세요.
    sparse_only_ids = None  # TODO: BM25 Top 결과에만 있는 id를 구하세요.

    if shared_ids is None or dense_only_ids is None or sparse_only_ids is None:
        print("TODO: shared_ids, dense_only_ids, sparse_only_ids를 완성하세요.")
        break

    print("Dense Top ID:", dense_ids)
    print("BM25 Top ID:", sparse_ids)
    print("Hybrid Top ID:", hybrid_ids)
    print("공통 ID:", shared_ids)
    print("Dense에만 있는 ID:", dense_only_ids)
    print("BM25에만 있는 ID:", sparse_only_ids)

    preview_results("Dense Only", dense_docs)
    preview_results("Sparse Only", sparse_docs)
    preview_results("Hybrid", hybrid_docs)


<details>
<summary>정답 코드 보기</summary>

아래 정답은 같은 질문을 Dense, BM25, Hybrid에 각각 넣고 Top 결과 ID를 비교합니다.  
ID를 비교하면 두 검색기가 같은 후보를 찾았는지, 서로 다른 후보를 보완하는지 보기 쉽습니다.

```python
comparison_queries = {
    "정확한 용어 중심 질문": "반대매매와 추가담보 납부 통지에 대해 투자자가 확인해야 할 사항은 무엇인가?",
    "풀어쓴 표현 중심 질문": "담보가 부족해서 주식이 자동으로 팔리기 전에 고객이 뭘 확인해야 하나요?",
}
top_k = 4  # 너무 많이 보면 비교가 어려우므로 상위 4개만 봅니다.

for label, query in comparison_queries.items():
    print(f"\n===== {label} =====")
    print("질문:", query)

    dense_docs = dense_retriever.invoke(query)  # 의미가 비슷한 청크를 찾습니다.
    sparse_docs = bm25_retriever.invoke(query)  # 정확한 키워드가 들어간 청크를 찾습니다.
    hybrid_docs = hybrid_rrf.invoke(query)  # Dense와 BM25 결과를 순위 기반으로 합칩니다.

    dense_ids = [doc.metadata.get("id") for doc in dense_docs[:top_k]]  # Dense 상위 결과 ID입니다.
    sparse_ids = [doc.metadata.get("id") for doc in sparse_docs[:top_k]]  # BM25 상위 결과 ID입니다.
    hybrid_ids = [doc.metadata.get("id") for doc in hybrid_docs[:top_k]]  # Hybrid 상위 결과 ID입니다.

    shared_ids = [doc_id for doc_id in dense_ids if doc_id in sparse_ids]  # 두 방식이 모두 찾은 후보입니다.
    dense_only_ids = [doc_id for doc_id in dense_ids if doc_id not in sparse_ids]  # Dense에만 잡힌 후보입니다.
    sparse_only_ids = [doc_id for doc_id in sparse_ids if doc_id not in dense_ids]  # BM25에만 잡힌 후보입니다.

    print("Dense Top ID:", dense_ids)
    print("BM25 Top ID:", sparse_ids)
    print("Hybrid Top ID:", hybrid_ids)
    print("공통 ID:", shared_ids)
    print("Dense에만 있는 ID:", dense_only_ids)
    print("BM25에만 있는 ID:", sparse_only_ids)

    preview_results("Dense Only", dense_docs)
    preview_results("Sparse Only", sparse_docs)
    preview_results("Hybrid", hybrid_docs)
```

정확한 용어 중심 질문에서는 BM25가 강하게 반응할 수 있고, 풀어쓴 표현 중심 질문에서는 Dense가 의미를 따라가는 데 유리할 수 있습니다. Hybrid는 두 후보군을 함께 모아 놓고 비교할 때 의미가 있습니다.
</details>


**관찰 포인트**

- 어떤 방식이 가장 관련성 높은 청크를 포함하는가?  
- Dense-only에서는 누락되지만 Sparse에서는 잡히는 내용이 있는가?  
- Hybrid가 두 방식의 장점을 잘 결합했는가?

#### 정리 — Retrieval 단계

- Dense, Sparse, Hybrid는 모두 **Retrieval 단계에서 후보를 모으는 전략**입니다.
- 이 단계의 핵심 목표는 **정답 가능성이 높은 문서를 빠뜨리지 않는 것**입니다.
- RRF도 Post-Retrieval이 아니라, **검색 결과를 합치는 Retrieval 단계의 fusion 기법**입니다.



### 2.4 후보군 점검과 Reranking

앞 절까지는 retrieval 단계에서 정답 후보를 넓게 모았습니다.  
이제는 이렇게 가져온 후보를 그대로 LLM에 넣지 않고, **어떤 후보를 남기고 어떻게 정리할지**를 보는 단계로 넘어갑니다.

이 구간은 **Post-Retrieval**에 해당합니다.  
즉, 검색이 끝난 뒤 후보 문서를 다시 읽고, 순서를 조정하거나, 불필요한 부분을 줄이거나, 중복을 제거하는 단계입니다.

여기서 볼 대표 기법은 다음과 같습니다.

| 기법 | 하는 일 |
|---|---|
| **Reranking** | 상위 후보를 다시 읽고 더 관련성 높은 순서로 재정렬 |
| **Filtering** | 관련성이 낮은 후보를 제거 |
| **Compression** | 문서 안에서 필요한 부분만 남김 |
| **Deduplication** | 비슷한 후보의 반복을 줄임 |

반대로 **RRF**는 이 단계의 기법이 아니라, retrieval 단계에서 여러 검색 결과를 합치는 **fusion 기법**입니다.

이 구간에서 먼저 확인할 것은 세 가지입니다.
- 상위 후보에 정답 단서가 들어왔는가
- 후보가 너무 길거나 중복되지 않는가
- 기본 retrieval만으로 충분한가, 아니면 reranking 같은 후처리가 필요한가

즉, 이 구간의 관심사는 **무엇을 찾을까**보다, **가져온 후보를 어떻게 더 쓰기 좋게 만들까**에 있습니다.


#### 2.4.1 Reranking

앞에서 retrieval 단계에서 Dense, Sparse, Hybrid로 후보를 먼저 모았습니다.  
이렇게 모은 상위 후보는 이미 어느 정도 관련 있지만, 질문과의 직접 관련도 순서가 완벽하지 않을 수 있습니다.

이럴 때 쓰는 단계가 **Reranking**입니다.  
즉, retrieval이 모아온 상위 후보를 대상으로 질문과 문서를 다시 비교해 순서를 더 정확하게 다듬습니다.

여기서 먼저 구분할 것이 **bi-encoder**와 **cross-encoder**입니다.

| 방식 | 입력 방식 | 장점 | 한계 | 주로 쓰는 위치 |
|---|---|---|---|---|
| **Bi-Encoder** | 질문과 문서를 따로 임베딩 | 빠르고 전체 문서 검색에 적합 | 질문-문서의 세밀한 상호작용을 덜 봄 | 1차 Retrieval |
| **Cross-Encoder** | 질문과 문서를 한 쌍으로 함께 입력 | 관련도 판단이 더 정밀함 | 후보마다 모델을 다시 호출해 느림 | Reranking |

<img src="image/bi_cross_encoder_reranking.svg" width="820">

대표적으로 **cross-encoder**는 `query`와 `document`를 한 쌍으로 함께 읽고 관련도를 다시 점수화합니다.  
retrieval보다 느리기 때문에, 보통은 문서 전체가 아니라 **좁혀진 상위 후보 몇 개에만 선택적으로 적용**합니다.

실무에서는 이렇게 다시 매긴 점수를 기준으로 **상위 일부만 남겨 최종 컨텍스트 후보로 사용**하는 경우가 많습니다.  
즉, retrieval로 넓게 모은 뒤 reranking으로 순서를 다듬고, 그중 앞쪽 몇 개를 다음 단계로 넘기는 흐름입니다.

정리하면, retrieval은 **후보를 넓게 모으는 단계**, reranking은 **그 후보의 순서를 다시 정해 상위 일부를 추리는 단계**입니다.  
그다음에는 아래에서 보듯이, 남은 후보에서 중복을 줄이거나 필요한 부분만 남기는 **Filtering / Compression** 단계로 이어질 수 있습니다.


#### 2.4.2 Cross-Encoder Reranking 기본 예시

아래에서는 Hugging Face에 공개된 한국어 지원 reranker인 `dragonkue/bge-reranker-v2-m3-ko`를 사용합니다.  
이 모델은 multilingual `bge-reranker-v2-m3`를 기반으로 한국어에 맞게 조정된 cross-encoder 모델입니다.

첫 실행에서는 모델을 내려받기 때문에 시간이 걸릴 수 있습니다. CPU 환경에서 느리면 `candidate_docs` 개수를 4개 정도로 줄여도 됩니다.


**Cross-Encoder로 후보 재정렬**

Hybrid가 모은 후보를 질문과 다시 비교해 관련도가 높은 순서로 바꿉니다.


In [ ]:
import textwrap  # 긴 검색 결과를 짧게 보여줄 때 씁니다.
import torch  # sigmoid 활성화 함수를 지정할 때 씁니다.
from sentence_transformers import CrossEncoder  # Hugging Face cross-encoder reranker를 쉽게 불러옵니다.

RERANKER_MODEL_NAME = "dragonkue/bge-reranker-v2-m3-ko"  # 한국어에 맞게 조정된 공개 reranker입니다.
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,  # 질문-문서 쌍을 다시 평가할 모델 이름입니다.
    activation_fn=torch.nn.Sigmoid(),  # 점수를 0~1 범위로 해석하기 쉽게 만듭니다.
)

def cross_encoder_rerank(query, docs, top_k=5):  # 후보 문서를 질문과 다시 비교해 재정렬합니다.
    pairs = [[query, doc.page_content] for doc in docs]  # cross-encoder는 질문-문서 쌍을 입력으로 받습니다.
    scores = reranker.predict(pairs)  # 각 후보 문서의 관련도 점수를 계산합니다.
    ranked = sorted(zip(docs, scores), key=lambda item: float(item[1]), reverse=True)  # 점수가 높은 문서를 앞으로 보냅니다.
    return ranked[:top_k]

def preview_rerank_item(rank, doc, score=None):  # 재정렬 결과를 읽기 쉽게 출력합니다.
    page = doc.metadata.get("page")
    page_label = f"p.{page + 1}" if isinstance(page, int) else "page unknown"
    score_label = f" | score={float(score):.4f}" if score is not None else ""
    preview = textwrap.shorten(doc.page_content.replace("\n", " "), width=180, placeholder=" ...")
    print(f"{rank}. id={doc.metadata.get('id')} | {page_label}{score_label}")
    print(preview)

rerank_query = "금융투자회사 직원이 손실을 보전해 주겠다고 약속하면 왜 문제가 되는가?"
candidate_docs = hybrid_rrf.invoke(rerank_query)[:8]  # Hybrid Retrieval로 먼저 후보를 넓게 모읍니다.
reranked_docs = cross_encoder_rerank(rerank_query, candidate_docs, top_k=5)  # 상위 후보를 다시 정렬합니다.

print("=== Reranking 전: Hybrid Retrieval 순서 ===")
for rank, doc in enumerate(candidate_docs[:5], start=1):
    preview_rerank_item(rank, doc)
    print("---")

print("\n=== Reranking 후: Cross-Encoder 재정렬 순서 ===")
for rank, (doc, score) in enumerate(reranked_docs, start=1):
    preview_rerank_item(rank, doc, score)
    print("---")


### 2.5 Context Filtering & Compression (Post-Retrieval)

#### 2.5.1 왜 Filtering & Compression이 중요한가?

LLM은 검색된 문서를 많이 넣는다고 항상 더 잘 답하지 않습니다.  
후보 문서를 그대로 넣으면 길이, 중복, 잡음 때문에 오히려 핵심 근거를 놓칠 수 있습니다.

| 문제 | 왜 생기나 |
|---|---|
| **컨텍스트가 너무 길어짐** | 컨텍스트 창에는 한계가 있고, 길어질수록 비용과 처리 부담도 커집니다. |
| **질문과 직접 관계없는 내용이 섞임** | 부록, 머리말, 반복 설명 같은 정보가 attention을 분산시켜 필요한 근거를 찾기 어려워집니다. |
| **관련 문서 안에서도 핵심 근거를 찾기 어려움** | 문서 전체는 관련 있어도, 답에 필요한 문장 몇 개만 흩어져 있으면 긴 문맥 속에서 놓치기 쉽습니다. |

Filtering과 Compression은 이런 문제를 줄여 **LLM이 실제 답변에 필요한 정보만 더 선명하게 보게 하는 과정**입니다.


#### 2.5.2 Context Filtering 기법

| 기법 | 무엇을 줄이는가 | 대표 방법 |
|---|---|---|
| **Relevance Filtering** | 질문과 직접 관련 없는 문서 | 점수 cutoff, LLM 기반 관련성 판별 |
| **Top-N / Score Cutoff** | 관련도 낮은 하위 후보 | reranker 기준 상위 N개만 유지, 점수 threshold 적용 |
| **Deduplication** | 같은 내용을 반복하는 문서 | 텍스트/임베딩 유사도 기반 제거 |

Reranking은 순서를 다시 정하는 단계이고, Filtering은 그 결과를 바탕으로 하위 후보를 버려 최종 컨텍스트를 좁히는 데 함께 쓰일 수 있습니다.

문서 안에서 필요한 부분만 남기는 방식은 아래 Compression 단계에서 함께 봅니다.


#### 기본 예시: 검색 결과 중복 제거 (Deduplication)

아래 예제는 2.5.2 표에서 본 Deduplication을 직접 적용해 보는 코드입니다.

retrieval 이후에는 완전히 같은 청크가 아니어도, 비슷한 내용을 반복하는 청크가 여러 개 남을 수 있습니다.

Deduplication은 이런 유사 후보를 줄여, 검색 결과가 한 내용으로만 겹치지 않도록 정리하는 단계입니다.

핵심은 관련도를 다시 판단하는 것이 아니라, **100% 같은 중복만이 아니라 내용이 많이 겹치는 청크까지 함께 줄이는 것**입니다.


**중복 후보 줄이기**

검색 결과끼리 내용이 많이 겹치는지 임베딩 유사도로 비교해 반복 후보를 줄입니다.


In [ ]:
import numpy as np  # 벡터 배열 계산과 유사도 계산에 쓰는 수치 연산 라이브러리입니다.
from langchain_openai import OpenAIEmbeddings  # 텍스트를 의미 기반 숫자 벡터로 바꿉니다.

# 검색 결과끼리 너무 비슷한 청크를 줄이기 위해 다시 임베딩합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# retrieval이 끝난 뒤, 내용이 거의 같은 청크를 제거합니다.
def dedupe_after_retrieval(docs, threshold=0.90):  # threshold 이상으로 비슷한 청크를 중복으로 봅니다.
    if not docs:
        return [], []  # 비교할 문서가 없으면 바로 끝냅니다.

    texts = [d.page_content for d in docs]  # 중복 비교에 쓸 본문만 꺼냅니다.
    vectors = np.array(embeddings.embed_documents(texts))  # 각 문서를 임베딩 벡터로 바꿉니다.

    # 코사인 유사도를 쓰기 위해 벡터 길이를 1로 맞춥니다.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / (norms + 1e-12)

    keep_indices = []  # 남길 문서의 위치를 기록합니다.
    removed_indices = set()  # 중복으로 제거할 문서의 위치를 기록합니다.

    for i in range(len(docs)):
        if i in removed_indices:
            continue  # 이미 제거 대상으로 표시된 문서는 건너뜁니다.

        keep_indices.append(i)  # 현재 문서는 기준 문서로 남깁니다.
        sims = np.dot(vectors[i], vectors[i+1:].T)  # 현재 문서와 뒤쪽 문서의 유사도를 한 번에 계산합니다.
        for offset, sim in enumerate(sims, start=i+1):
            if sim >= threshold:  # 기준값보다 유사하면 뒤쪽 문서를 중복으로 처리합니다.
                removed_indices.add(offset)  # 너무 비슷하면 뒤쪽 문서를 제거 대상으로 표시합니다.

    kept_docs = [docs[i] for i in keep_indices]  # 최종적으로 남길 문서입니다.
    removed_docs = [docs[i] for i in sorted(removed_indices)]  # 중복으로 제거된 문서입니다.
    return kept_docs, removed_docs

# 같은 주제의 비슷한 규칙 청크가 많이 나오는 질문을 예시로 씁니다.
dedupe_query = "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?"
dense_candidates = dense_retriever.invoke(dedupe_query)[:4]  # Dense에서 상위 후보를 가져옵니다.
sparse_candidates = bm25_retriever.invoke(dedupe_query)[:4]  # Sparse에서 상위 후보를 가져옵니다.

# 어떤 검색기에서 온 문서인지 표시해 두면 제거 결과를 읽기 쉽습니다.
for doc in dense_candidates:
    doc.metadata["retrieval_source"] = "dense"
for doc in sparse_candidates:
    doc.metadata["retrieval_source"] = "bm25"

candidate_docs = dense_candidates + sparse_candidates  # 두 검색기의 후보를 한데 모읍니다.
deduped_docs, removed_docs = dedupe_after_retrieval(candidate_docs, threshold=0.88)  # 0.88 이상 비슷한 후보를 줄입니다.

print("중복 제거 전 개수:", len(candidate_docs))
print("중복 제거 후 개수:", len(deduped_docs))

print("\n=== 제거된 문서 ===")
for doc in removed_docs[:3]:
    print(f"[id={doc.metadata.get('id')}] source={doc.metadata.get('retrieval_source')}")
    print(doc.page_content[:160])
    print("---")

print("\n=== 중복 제거 후 남은 문서 ===")
for doc in deduped_docs[:5]:
    print(f"[id={doc.metadata.get('id')}] source={doc.metadata.get('retrieval_source', 'kept')}")
    print(doc.page_content[:160])
    print("---")

if not removed_docs:
    print("이번 질문에서는 제거된 문서가 많지 않지만, 비슷한 규칙 목록이 반복될 때는 차이가 더 커질 수 있습니다.")


#### 2.5.3 Context Compression 기법

Context Compression은 결국 **질문에 필요한 정보만 남기도록 문서를 줄이는 과정**이라고 이해하면 됩니다.
구현 방식은 크게 두 가지로 나눌 수 있습니다.

| 방식 | 무엇을 하느냐 |
|---|---|
| **Extractive Compression** | 원문에서 질문과 관련된 문장이나 구절만 뽑아 남김 |
| **Abstractive Compression** | 질문에 필요한 내용을 짧게 다시 표현해 새 문장으로 요약 |

즉, 둘 다 질문 기준으로 문맥을 줄이는 작업이고,
차이는 **원문을 그대로 남기느냐**, 아니면 **새 문장으로 다시 요약하느냐**에 있습니다.


#### 기본 예시: 질문 기준 문맥 압축

아래 예제는 질문에 필요한 정보만 골라 새 문장으로 요약합니다.  
즉, 질문 기준으로 문맥을 줄이는 **요약형(Abstractive) Compression** 예시입니다.


**질문 기준 문맥 압축**

검색된 문서에서 질문에 필요한 내용만 짧게 요약해 컨텍스트를 줄입니다.


In [ ]:
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.
from langchain_core.documents import Document  # 텍스트와 메타데이터를 함께 담는 문서 객체입니다.

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 질문 기준 문맥 압축에 사용할 생성 모델입니다.

# 질문 기준으로 필요한 내용만 새 문장으로 요약하는 프롬프트입니다.
compress_prompt = ChatPromptTemplate.from_template("""
당신은 '문서 요약 전문가'입니다.
주어진 [문서]에서 사용자의 [질문]에 답변하는 데 필요한 **핵심 정보**만 추출하여 요약하세요.

질문: {query}

문서:
{content}

출력 규칙:
- 질문과 직접 관련된 내용만 3~5줄로 요약
- 배경 설명, 문서 전체 요약은 제외
- 만약 문서에 질문과 관련된 내용이 전혀 없다면 "관련 정보 없음"이라고만 출력
""")

compress_chain = compress_prompt | llm  # 질문과 문서를 넣으면 필요한 내용만 요약하는 체인입니다.

def compressed_retrieve(dense_retriever, query: str, k: int = 4):  # 검색 뒤 각 문서를 질문 기준으로 요약 압축합니다.
    dense_docs = dense_retriever.invoke(query)[:k]  # 먼저 상위 k개 문서를 가져옵니다.
    compressed_docs = []  # 요약된 문서를 다시 담아 둘 리스트입니다.

    for doc in dense_docs:  # 상위 문서를 하나씩 압축합니다.
        summary = compress_chain.invoke({"query": query, "content": doc.page_content}).content  # 질문에 필요한 내용을 새 문장으로 요약합니다.
        compressed_docs.append(Document(page_content=summary, metadata=doc.metadata))  # 요약 결과도 Document 형태로 맞춥니다.

    return compressed_docs

compression_query = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"
raw_docs = dense_retriever.invoke(compression_query)[:3]  # 압축 전 원본 문서를 봅니다.
compressed_docs = compressed_retrieve(dense_retriever, compression_query, k=3)  # 같은 문서를 질문 기준 압축 버전으로 만듭니다.

print("=== 압축 전 ===")
for i, doc in enumerate(raw_docs, 1):
    print(f"[{i}] id={doc.metadata.get('id')} / 길이={len(doc.page_content)}")
    print(doc.page_content[:180])  # 길이와 내용이 얼마나 줄었는지 비교합니다.
    print("---")

print("\n=== 압축 후 ===")
for i, doc in enumerate(compressed_docs, 1):
    print(f"[{i}] id={doc.metadata.get('id')} / 길이={len(doc.page_content)}")
    print(doc.page_content[:180])  # 길이와 내용이 얼마나 줄었는지 비교합니다.
    print("---")


##### 실습 문제 4: 문맥 압축 프롬프트 설계하기

Context Compression은 단순 요약이 아닙니다.  
질문에 필요한 근거는 남기고, 질문과 무관한 문서는 과감히 버릴 수 있어야 합니다.

이 문제에서는 관련 문서와 무관한 문서를 함께 넣습니다.  
좋은 압축 프롬프트라면 관련 문서는 짧게 압축하고, 무관한 문서는 `관련 정보 부족`으로 처리해야 합니다.

**작업**

- `practice_compress_prompt`를 직접 작성합니다.
- 질문과 직접 관련된 문장만 남기도록 지시합니다.
- 원문에 없는 내용을 새로 만들지 않도록 제한합니다.
- 무관한 문서는 `관련 정보 부족`이라고만 출력하게 합니다.
- 압축 전후를 비교합니다.


In [ ]:
practice_compression_query = "보이스피싱을 의심할 때 바로 해야 할 일은 무엇인가?"
relevant_docs = dense_retriever.invoke(practice_compression_query)[:1]  # 질문과 관련된 후보입니다.
irrelevant_docs = dense_retriever.invoke("펀드 가입 시 투자자 유형 분류 기준은 무엇인가?")[:1]  # 일부러 섞는 무관한 후보입니다.
practice_raw_docs = relevant_docs + irrelevant_docs

print("=== 압축 전 후보 ===")
for i, doc in enumerate(practice_raw_docs, 1):
    print(f"[{i}] 길이={len(doc.page_content)}")
    print(doc.page_content[:180])
    print("---")

practice_compress_prompt = None  # TODO: ChatPromptTemplate.from_template(...)로 압축 프롬프트를 작성하세요.

if practice_compress_prompt is None:
    print("TODO: practice_compress_prompt를 작성한 뒤 압축 전후를 비교하세요.")
else:
    practice_compress_chain = practice_compress_prompt | llm

    practice_compressed_docs = []
    for doc in practice_raw_docs:
        summary = practice_compress_chain.invoke({"query": practice_compression_query, "content": doc.page_content}).content  # 질문 기준으로 압축합니다.
        practice_compressed_docs.append(Document(page_content=summary, metadata=doc.metadata))

    for i, (raw_doc, compressed_doc) in enumerate(zip(practice_raw_docs, practice_compressed_docs), 1):
        print(f"===== 문서 {i} =====")
        print("압축 전 길이:", len(raw_doc.page_content))
        print(raw_doc.page_content[:180])
        print("\n압축 후 길이:", len(compressed_doc.page_content))
        print(compressed_doc.page_content[:220])
        print()


<details>
<summary>정답 코드 보기</summary>

아래 정답은 관련 문서와 무관한 문서를 구분하도록 압축 규칙을 분명히 둡니다.

```python
practice_compress_prompt = ChatPromptTemplate.from_template("""
주어진 [문서]에서 [질문]에 답하는 데 필요한 근거만 남겨 압축하세요.

[압축 규칙]
1. 질문과 직접 관련된 내용만 남깁니다.
2. 원문에 없는 내용은 추가하지 않습니다.
3. 숫자, 조건, 금지사항, 행동 지침은 가능하면 그대로 유지합니다.
4. 질문과 관련된 근거가 부족하면 "관련 정보 부족"이라고만 답합니다.
5. 관련 근거가 있으면 2~4줄로 작성합니다.

질문: {query}

문서:
{content}

압축 결과:
""")  # 관련 근거만 남기고 무관한 문서는 버리게 합니다.

practice_compress_chain = practice_compress_prompt | llm

practice_compressed_docs = []
for doc in practice_raw_docs:
    summary = practice_compress_chain.invoke({
        "query": practice_compression_query,
        "content": doc.page_content,
    }).content  # 각 문서를 같은 질문 기준으로 압축합니다.
    practice_compressed_docs.append(Document(page_content=summary, metadata=doc.metadata))

for i, (raw_doc, compressed_doc) in enumerate(zip(practice_raw_docs, practice_compressed_docs), 1):
    print(f"===== 문서 {i} =====")
    print("압축 전 길이:", len(raw_doc.page_content))
    print(raw_doc.page_content[:180])
    print("\n압축 후 길이:", len(compressed_doc.page_content))
    print(compressed_doc.page_content[:220])
    print()
```
</details>


**관찰 포인트**
- 검색된 문서보다 압축된 문서의 길이가 크게 줄어 있는가?  
- 질문과 직접 관련된 내용만 남고 주변 설명은 줄어들었는가?  
- 불필요한 서론/부록/표 등이 사라졌는가?

#### 정리

- **RRF**는 Dense와 Sparse 결과를 결합해 순서를 다시 정하는 retrieval 단계의 fusion입니다.
- **Filtering**은 불필요한 문서를 덜어내는 과정입니다.
- **Deduplication**은 Filtering 안에서 서로 비슷한 후보를 줄여 문맥 밀도를 높이는 방법입니다.
- **Compression**은 남길 문서 안에서 핵심만 짧게 만드는 과정입니다.


### Multi-Representation Indexing

하나의 문서를 하나의 표현으로만 저장할 필요는 없습니다.  
원문 그대로 검색했을 때 질문과 문서가 잘 만나지 않는다면, 검색용 표현을 따로 만들어 볼 수 있습니다.

특히 다음 상황에서 도움이 됩니다.

- 청크가 길고 표, 목록, 예외, 부가 설명이 섞여 핵심 주제가 묻힐 때
- 조항, 약관, 매뉴얼처럼 문장 구조가 딱딱해 사용자 질문과 말투가 크게 다를 때
- `이 경우`, `해당 상품`, `위 절차`처럼 대명사, 생략, 앞뒤 문맥 의존이 많을 때
- 문서의 전문용어와 사용자의 표현이 다를 때  
  예: `직원이 알아서 매매` ↔ `일임매매`, `손실을 메워주겠다는 약속` ↔ `손실보전`
- 한 문서가 여러 질문에 답할 수 있어, 질문형 표현을 함께 저장하면 찾기 쉬울 때

다만 검색용 표현은 답변 근거가 아니라 **문서를 잘 찾기 위한 표지**입니다.  
답변을 만들 때는 검색용 표현이 아니라 원문 청크를 근거로 사용해야 합니다.

| 표현 방식 | 무엇을 저장하나 | 기대 효과 |
|---|---|---|
| 원문 청크 | 실제 본문 그대로 | 답변 근거로 바로 쓰기 좋습니다. |
| surrogate text | 짧은 설명, 요약, 핵심 문장 | 긴 원문의 잡음을 줄이고 핵심 주제와 용어를 드러낼 수 있습니다. |
| 질문형 표현 | 이 문서가 답하는 질문 | 사용자 질문과 질문형 표현을 비교해 말투 차이를 줄일 수 있습니다. |
| parent-child | 검색은 작은 청크, 전달은 큰 문맥 | 검색 정확도와 문맥 보존을 함께 노릴 수 있습니다. |

여기서는 가장 단순한 형태로 **surrogate text**를 만들어 봅니다.


아이디어는 단순합니다.

- 검색은 짧고 또렷한 대표 표현으로 수행하고
- 답변 생성에는 원문을 다시 꺼내 쓰는 방식입니다.

이 구조는 요약 인덱스, parent-child, Graph RAG 노드 요약문으로도 이어지는 기본 패턴입니다.

<img src="image/multi_representation_indexing.svg" width="820">


**검색용 대표 표현 만들기**

원문 청크를 그대로 검색에 쓰지 않고, 검색에 잘 걸리는 짧은 대표 표현을 따로 만듭니다.  
검색 결과에는 원문도 함께 보관해 두었다가, 답변 생성 단계에서는 원문을 다시 사용합니다.

아래 코드는 원리를 보기 위한 최소 예시입니다. `key_terms`와 `if` 문으로 대표 표현을 만들지만, 현업에서는 보통 다음 정보를 함께 설계합니다.

- 문서나 섹션 요약
- 핵심 엔터티와 용어
- 섹션 제목과 문서 구조
- 이 문서가 답할 수 있는 질문형 표현
- LLM이 생성한 대표 문장

핵심은 규칙을 정교하게 많이 쓰는 것이 아니라, **검색용 표현과 답변용 원문을 분리하는 것**입니다.


In [ ]:
import re  # 청크 텍스트를 한 줄로 정리할 때 씁니다.
import textwrap  # 원문 미리보기를 짧게 줄일 때 씁니다.
from uuid import uuid4  # 재실행할 때 임시 컬렉션 이름이 겹치지 않게 만듭니다.
from langchain_core.documents import Document  # 검색용 대표 표현도 Document로 저장합니다.

key_terms = [
    '보이스피싱', '금융사기', 'OTP', '비밀번호', '모바일',  # 검색에 잘 걸릴 핵심 용어들입니다.
    '분쟁', '판례', '투자자', '공시', '반대매매'
]

def make_surrogate_text(doc):  # 원문 청크를 검색용 대표 문장으로 바꿉니다.
    flat = re.sub(r'\s+', ' ', doc.page_content).strip()  # 줄바꿈과 공백을 한 줄 형태로 정리합니다.
    found_terms = [term for term in key_terms if term in flat][:5]  # 핵심 용어가 있으면 앞부분에 드러냅니다.
    hints = []  # 사용자가 물을 법한 질문형 표현을 앞에 붙입니다.
    if any(term in flat for term in ['모바일', '모바일폰', '휴대폰', 'OTP', '잠금', '공식 배포처']):
        hints.append('이 문서는 모바일 금융거래 보안 수칙 질문에 답한다.')
    if any(term in flat for term in ['보이스피싱', '피해구제', '지급정지']):
        hints.append('이 문서는 보이스피싱 피해 대응 절차 질문에 답한다.')
    if '반대매매' in flat:
        hints.append('이 문서는 반대매매와 추가담보 납부 질문에 답한다.')
    keyword_line = f"핵심 키워드: {', '.join(found_terms)}" if found_terms else '핵심 키워드: 없음'
    summary_line = f"검색용 요약: {flat[:260]}"
    return '\n'.join(hints + [keyword_line, summary_line])

base_docs = split_docs[20:140]  # 목차 이후의 본문 일부만 사용합니다.
surrogate_docs = [
    Document(
        page_content=make_surrogate_text(doc),  # 검색용 대표 표현을 본문으로 저장합니다.
        metadata={
            **doc.metadata,  # 원래 메타데이터는 그대로 유지합니다.
            'original_text': doc.page_content,  # 생성 단계에서 쓸 원문도 함께 보관합니다.
        },
    )
    for doc in base_docs
]

surrogate_store = Chroma.from_documents(
    documents=surrogate_docs,  # 원문이 아니라 검색용 대표 표현을 저장합니다.
    embedding=embeddings,  # 대표 표현도 기존 검색과 같은 임베딩 모델로 벡터화합니다.
    collection_name=f"surrogate_docs_{uuid4().hex[:8]}",  # 재실행해도 기존 임시 컬렉션과 섞이지 않게 합니다.
)  # 대표 표현으로 새 벡터 저장소를 만듭니다.
len(surrogate_docs)  # 저장한 대표 표현 개수를 확인합니다.


**대표 표현으로 검색하고 원문 확인**

짧은 대표 표현으로 검색한 뒤, 답변에 쓸 원문이 함께 연결되는지 확인합니다.


In [ ]:
query = '모바일 투자 시 주의할 점을 알려줘'  # 구어체 질문 예시입니다.
surrogate_hits = surrogate_store.similarity_search(query, k=3)  # 대표 표현 저장소에서 상위 3개를 찾습니다.

for idx, hit in enumerate(surrogate_hits, start=1):
    print(f'===== hit {idx} =====')  # 몇 번째 검색 결과인지 표시합니다.
    print('[search text]')
    print(hit.page_content)  # 검색에 실제로 쓰인 대표 표현을 봅니다.
    print('\n[original text]')
    print(textwrap.shorten(hit.metadata['original_text'].replace('\n', ' '), width=300, placeholder=' ...'))  # 생성 단계에 넘길 원문도 함께 확인합니다.
    print()


##### 실습 문제 5: 2일차 종합 RAG 파이프라인 구축하기

지금까지는 각 기법을 작은 예제로 나누어 확인했습니다.  
이번 문제는 완성된 코드를 따라 치는 문제가 아니라, 필요한 조건만 보고 직접 RAG 파이프라인을 조립하는 문제입니다.

**상황**

금융투자 안내서를 기반으로 고객 상담 챗봇을 만든다고 가정합니다.  
사용자는 문서의 공식 용어를 모른 채 불안하거나 모호한 말투로 질문할 수 있습니다.  
챗봇은 근거 문서를 찾아 차분한 상담 담당자 말투로 답해야 합니다.

**제공 정보**

- ChromaDB 상대경로: `./chroma_index_kofia_guide_cleaned`
- ChromaDB 컬렉션 이름: `kofia_guide_cleaned`
- 임베딩 모델: `text-embedding-3-small`
- 답변 생성 모델: `gpt-5-nano`
- 기준 문서: `data/금융투자협회_투자길라잡이_2018.pdf`

**구현 조건**

- `answer(question)` 형태의 함수를 직접 만듭니다.
- ChromaDB를 불러와 검색기를 구성합니다.
- Multi-Query, HyDE, Hybrid Retrieval, RRF, Compression 중 최소 하나를 직접 적용합니다.
- Deduplication 또는 Reranking 중 하나 이상을 적용해 후보 문서를 한 번 더 정리합니다.
- 검색 결과를 답변 프롬프트에 넣어 최종 답변을 생성합니다.
- `evaluation_items`, 정답표, 출처 페이지를 직접 사용하지 않습니다.

**답변 출력 원칙**

- 첫 문장은 결론부터 씁니다.
- 답변은 5~7문장 정도로 간결하게 씁니다.
- 근거는 검색된 문서 안에서만 사용합니다.
- 검색 문맥에 직접 있는 내용만 말하고, 일반적인 조언을 덧붙이지 않습니다.
- 문서에서 확인되지 않으면 `제공된 안내서에서는 확인되지 않습니다.`라고 답합니다.
- 말투는 차분한 금융회사 상담 담당자처럼 씁니다.
- 과장된 안심, 단정적인 법률 판단, 문서에 없는 조언은 피합니다.
- 후속 질문을 유도하는 문장은 쓰지 않습니다.
- 마지막 줄에 `확인한 근거 페이지: p.xx, p.yy` 형식으로 사용한 페이지를 적습니다.

**점검 질문**

- `직원이 계좌를 알아서 굴려준다는데 맡겨도 괜찮나요?`
- `손실을 보전해주겠다는 약속을 받으면 안전한가요?`
- `모바일로 투자할 때 비밀번호나 OTP는 어떻게 관리해야 하나요?`

**확인할 점**

- 검색 결과가 질문의 핵심 주제와 맞는가?
- Deduplication 또는 Reranking 뒤에 문맥 품질이 좋아졌는가?
- 답변 말투가 너무 딱딱한 문서 복사나 과한 상담 멘트로 치우치지 않았는가?
- 근거 페이지가 실제 답변 내용과 연결되는가?


In [ ]:
# 최소 시작 코드입니다. 아래 설정만 사용해 직접 파이프라인을 구성하세요.
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from sentence_transformers import CrossEncoder

load_dotenv(".env", override=True)

DB_PATH = "./chroma_index_kofia_guide_cleaned"
COLLECTION_NAME = "kofia_guide_cleaned"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"
GENERATOR_MODEL_NAME = "gpt-5-nano"

TEST_QUESTIONS = [
    "직원이 계좌를 알아서 굴려준다는데 맡겨도 괜찮나요?",
    "손실을 보전해주겠다는 약속을 받으면 안전한가요?",
    "모바일로 투자할 때 비밀번호나 OTP는 어떻게 관리해야 하나요?",
]

# TODO
# 1. ChromaDB를 불러오세요.
# 2. 검색 전략을 직접 구성하세요.
# 3. Deduplication 또는 Reranking을 적용하세요.
# 4. 상담 담당자 말투의 답변 프롬프트를 작성하세요.
# 5. answer(question) 함수를 만들고 TEST_QUESTIONS로 결과를 확인하세요.


<details>
<summary>정답 코드 보기</summary>

아래 정답은 한 가지 예시입니다.  
Dense 검색과 BM25 검색을 RRF로 합친 뒤, Cross-Encoder Reranker로 최종 문서를 고르는 구조입니다.  
정답용 코드이므로 각 줄의 역할을 주석으로 자세히 적었습니다.

```python
# 위 셀에서 import, DB 경로, 모델명, 테스트 질문을 이미 준비했다는 전제로 시작합니다.

# 1. ChromaDB를 불러옵니다.
# OpenAIEmbeddings는 ChromaDB에 저장된 벡터와 같은 임베딩 모델을 사용해야 합니다.
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)  # 질문을 벡터로 바꿀 임베딩 모델입니다.

# 이미 만들어진 ChromaDB를 연결합니다.
vectorstore = Chroma(
    persist_directory=DB_PATH,  # 벡터DB가 저장된 폴더입니다.
    collection_name=COLLECTION_NAME,  # 사용할 Chroma 컬렉션 이름입니다.
    embedding_function=embeddings,  # 검색 질문을 벡터화할 때 사용할 임베딩 함수입니다.
)

# Dense retriever는 질문과 의미가 비슷한 청크를 찾습니다.
dense_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}  # 의미 기반 검색에서 상위 8개 후보를 가져옵니다.
)


# 2. BM25Retriever에 넣을 문서를 ChromaDB에서 꺼냅니다.
# ChromaDB에는 원문 청크와 metadata가 함께 들어 있으므로, 이를 BM25용 문서로 다시 구성합니다.
store_items = vectorstore.get(
    include=["documents", "metadatas"]  # BM25 색인에 필요한 본문과 metadata만 가져옵니다.
)

# Chroma 내부 id가 있으면 중복 제거 기준으로 사용하고, 없으면 None으로 채웁니다.
store_ids = store_items.get("ids") or [None] * len(store_items["documents"])

# BM25Retriever.from_documents에 넘길 Document 목록을 담을 리스트입니다.
bm25_docs = []

# Chroma에서 꺼낸 id, 본문, metadata를 같은 순서로 하나씩 묶어 봅니다.
for doc_id, content, metadata in zip(
    store_ids,  # Chroma 내부 문서 id입니다.
    store_items["documents"],  # 실제 검색 대상이 되는 청크 본문입니다.
    store_items["metadatas"],  # page, source 같은 부가 정보입니다.
):
    metadata = dict(metadata or {})  # metadata가 None이어도 안전하게 dict로 바꿉니다.
    metadata["chroma_id"] = doc_id  # Dense 결과와 BM25 결과의 같은 청크를 알아보기 위한 값입니다.
    doc = Document(page_content=content, metadata=metadata)  # LangChain retriever가 쓰는 문서 형식으로 바꿉니다.
    bm25_docs.append(doc)  # BM25 색인에 넣을 문서 목록에 추가합니다.

# BM25Retriever는 키워드가 정확히 겹치는 문서를 찾는 데 유리합니다.
bm25_retriever = BM25Retriever.from_documents(bm25_docs)  # 위에서 만든 문서들로 BM25 색인을 만듭니다.
bm25_retriever.k = 8  # 키워드 기반 검색에서도 상위 8개 후보를 가져옵니다.


# 3. Dense와 BM25 결과를 RRF로 합칩니다.
# RRF는 서로 다른 검색 결과를 순위 기준으로 합치는 방법입니다.
def doc_key(doc):
    """같은 청크를 중복으로 넣지 않기 위한 기준을 만듭니다."""
    # 가장 먼저 Chroma 내부 id를 사용합니다.
    if doc.metadata.get("chroma_id"):
        return doc.metadata["chroma_id"]

    # Chroma id가 없으면 기존 metadata의 id를 사용합니다.
    if doc.metadata.get("id"):
        return doc.metadata["id"]

    # 둘 다 없으면 본문 앞부분을 임시 기준으로 사용합니다.
    return doc.page_content[:120]


def rrf_combine(result_lists, k=60):
    """여러 검색 결과 리스트를 RRF 점수로 합칩니다."""
    scores = {}  # 문서별 RRF 누적 점수를 담습니다.
    docs_by_key = {}  # 문서 key로 실제 Document 객체를 다시 찾기 위한 dict입니다.

    # Dense 결과와 BM25 결과처럼 여러 검색 결과 리스트를 차례로 봅니다.
    for docs in result_lists:
        # 각 검색 결과 안에서 1등, 2등, 3등 순위를 계산합니다.
        for rank, doc in enumerate(docs, start=1):
            key = doc_key(doc)  # 같은 문서인지 판단할 key를 만듭니다.
            docs_by_key[key] = doc  # 나중에 key로 Document를 다시 꺼낼 수 있게 저장합니다.
            scores[key] = scores.get(key, 0) + 1 / (k + rank)  # 순위가 높을수록 더 큰 점수를 더합니다.

    # RRF 점수가 높은 문서 key부터 정렬합니다.
    sorted_keys = sorted(scores, key=scores.get, reverse=True)

    # 정렬된 key를 다시 Document 객체로 바꿔 반환합니다.
    return [docs_by_key[key] for key in sorted_keys]


# 4. Reranker로 질문과 문서가 직접 잘 맞는지 다시 점수화합니다.
# 첫 검색은 후보를 넓게 모으는 단계이고, reranker는 최종 후보를 좁히는 단계입니다.
reranker = CrossEncoder("dragonkue/bge-reranker-v2-m3-ko")  # 한국어 문서 재정렬에 사용할 reranker입니다.


def rerank_docs(query, docs, top_k=5):
    """질문-문서 쌍을 다시 평가해 최종 문서를 고릅니다."""
    # 검색 결과가 비어 있으면 reranker에 넣을 것이 없으므로 빈 리스트를 반환합니다.
    if not docs:
        return []

    # CrossEncoder는 [질문, 문서] 쌍을 입력으로 받아 관련도 점수를 계산합니다.
    pairs = [[query, doc.page_content] for doc in docs]

    # 각 질문-문서 쌍의 관련도 점수를 계산합니다.
    scores = reranker.predict(pairs)

    # 점수가 높은 문서가 앞으로 오도록 정렬합니다.
    ranked = sorted(zip(docs, scores), key=lambda item: float(item[1]), reverse=True)

    # 답변 프롬프트에 넣을 최종 문서 수만 남깁니다.
    return [doc for doc, score in ranked[:top_k]]


def retrieve_docs(question):
    """Hybrid Retrieval, RRF, Reranking을 하나로 묶은 검색 함수입니다."""
    dense_docs = dense_retriever.invoke(question)  # 의미가 비슷한 후보를 가져옵니다.
    sparse_docs = bm25_retriever.invoke(question)  # 키워드가 겹치는 후보를 가져옵니다.
    fused_docs = rrf_combine([dense_docs, sparse_docs])  # 두 후보군을 RRF로 합치고 중복도 정리합니다.
    final_docs = rerank_docs(question, fused_docs, top_k=5)  # reranker로 최종 5개 문서를 고릅니다.
    return final_docs  # 답변 생성에 사용할 문서 목록입니다.


# 5. 답변에 넣을 문맥과 페이지 표시를 정리합니다.
def page_label(doc):
    """metadata의 page 값을 사람이 읽는 페이지 표기로 바꿉니다."""
    page = doc.metadata.get("page")  # PDF 로더가 저장해 둔 페이지 번호를 꺼냅니다.

    # page가 정수이면 0부터 시작하는 번호일 가능성이 높으므로 1을 더합니다.
    if isinstance(page, int):
        return f"p.{page + 1}"

    # page 정보가 없으면 페이지 표기를 만들지 않습니다.
    return None


def format_context(docs):
    """LLM이 근거를 구분해 읽을 수 있도록 페이지와 본문을 함께 묶습니다."""
    chunks = []  # 프롬프트에 넣을 문맥 조각들을 담습니다.

    # 최종 선택된 문서를 하나씩 프롬프트용 문자열로 바꿉니다.
    for doc in docs:
        label = page_label(doc)  # 이 문서의 페이지 표기를 만듭니다.
        prefix = f"[{label}] " if label else ""  # 페이지가 있으면 본문 앞에 붙입니다.
        chunks.append(prefix + doc.page_content)  # 페이지 표기와 본문을 함께 저장합니다.

    # 문서 사이를 빈 줄로 띄워 LLM이 청크 경계를 읽기 쉽게 합니다.
    return "\n\n".join(chunks)


def format_pages(docs):
    """답변 마지막 줄에 넣을 근거 페이지를 중복 없이 정리합니다."""
    pages = []  # 이미 추가한 페이지를 순서대로 담습니다.

    # 답변에 사용한 문서들의 page metadata를 확인합니다.
    for doc in docs:
        label = page_label(doc)  # p.30 같은 페이지 표기를 만듭니다.

        # 페이지 정보가 있고 아직 넣지 않은 페이지라면 추가합니다.
        if label and label not in pages:
            pages.append(label)

    # 페이지가 있으면 쉼표로 묶고, 없으면 확인 불가라고 표시합니다.
    return ", ".join(pages) if pages else "확인 불가"


# 6. 상담 담당자 말투로 답하게 하는 프롬프트입니다.
# 핵심은 검색된 문서 안에서만 답하게 하고, 말투와 금지사항을 분명히 주는 것입니다.
answer_prompt = ChatPromptTemplate.from_template("""
당신은 금융투자 안내서를 근거로 답하는 상담 담당자입니다.
아래 [검색 문맥] 안에서 확인되는 내용만 사용해 답변하세요.

[답변 원칙]
- 첫 문장은 결론부터 씁니다.
- 답변은 5~7문장 이내로 간결하게 씁니다.
- 차분하고 친절한 금융회사 상담 담당자 말투로 씁니다.
- 검색 문맥에 직접 있는 내용만 말하고, 일반적인 조언을 덧붙이지 않습니다.
- 문서에 있는 투자자 유의사항과 금지사항을 중심으로 설명합니다.
- 과장된 안심, 단정적인 법률 판단, 문서에 없는 조언은 하지 않습니다.
- 후속 질문을 유도하는 문장은 쓰지 않습니다.
- 근거가 부족하면 "제공된 안내서에서는 확인되지 않습니다."라고 답합니다.
- 마지막 줄의 근거 페이지는 코드에서 붙일 예정이므로 본문에서는 쓰지 않습니다.

[검색 문맥]
{context}

[질문]
{question}

답변:
""")

# 답변 생성을 담당할 LLM입니다.
llm = ChatOpenAI(model=GENERATOR_MODEL_NAME)

# 프롬프트 -> LLM -> 문자열 파서 순서로 답변 체인을 만듭니다.
answer_chain = answer_prompt | llm | StrOutputParser()


def answer(question):
    """질문을 받아 검색, reranking, 답변 생성을 한 번에 실행합니다."""
    docs = retrieve_docs(question)  # 질문과 관련된 최종 근거 문서를 찾습니다.

    # 검색된 근거가 없으면 문서에서 확인되지 않는다고 답합니다.
    if not docs:
        return "제공된 안내서에서는 확인되지 않습니다.\n확인한 근거 페이지: 확인 불가"

    context = format_context(docs)  # 최종 문서를 프롬프트에 넣을 문맥 문자열로 바꿉니다.
    pages = format_pages(docs)  # 답변 마지막에 붙일 근거 페이지 목록을 만듭니다.

    # 질문과 검색 문맥을 넣어 답변을 생성합니다.
    response = answer_chain.invoke({"question": question, "context": context}).strip()

    # LLM이 만든 답변 뒤에 코드가 계산한 근거 페이지를 붙입니다.
    return f"{response}\n확인한 근거 페이지: {pages}"


# 준비한 점검 질문을 하나씩 실행해 결과를 확인합니다.
for question in TEST_QUESTIONS:
    print("=" * 80)  # 질문별 출력 구분선입니다.
    print("질문:", question)  # 현재 확인 중인 질문입니다.
    print(answer(question))  # RAG 파이프라인이 만든 답변입니다.
    print()  # 다음 질문 출력과 간격을 둡니다.
```
</details>


## 3. 참고자료

| 기법 | 대표 참고 논문 | 메모 |
|---|---|---|
| Multi-Query / Query Expansion | Wang et al., *Query2doc: Query Expansion with Large Language Models* (EMNLP 2023) | LLM 기반 질의 확장 흐름 참고 |
| HyDE | Gao et al., *Precise Zero-Shot Dense Retrieval without Relevance Labels* (ACL 2023) | 가상 문서 기반 dense retrieval 참고 |
| RRF | Cormack et al., *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods* (SIGIR 2009) | 여러 검색 결과의 순위 기반 결합 참고 |
| Cross-encoder Reranking | Nogueira and Cho, *Passage Re-ranking with BERT* (2019) | query-document 쌍 재점수화 예시 |
| Korean Reranker | `dragonkue/bge-reranker-v2-m3-ko` model card | 한국어 지원 cross-encoder reranker 예제 모델 |
